## 0 · The Challenge

> **The mission**: Riverside House must turn a lightweight 19.7M-parameter model into a useful in-house editing assistant without sending any of its 619,000 confidential words to a public API.

**What we know so far:**

- Qwen2.5 can already produce simple English fiction on Riverside's CPU.
- Riverside can test every candidate with the same three acceptance probes: catalog fluency, instruction compliance, and editor preference.
- **But the base model fails the first probe:** it has never seen Riverside's unpublished stories.

**What's blocking us:**
The model has never experienced Riverside's corpus. Giving it the manuscripts may improve continuation, but that only practices prose. If it later ignores a bounded editor request, that failure must determine the next training experience. If it follows the request but chooses a needlessly verbose answer, that remaining failure determines the final one.

**What this chapter unlocks:**
A failure-driven training path. We will expose one failed probe, apply the smallest data-objective change that addresses it, then rerun the probes before adding another technique.

# LLM Fine-Tuning Deep Dive, Part 1 of 3: What Should the Model Learn?

> **The story:** Riverside will change the model's training experience only when an observed failure justifies it: unfamiliar catalog prose motivates continued pretraining, ignored requests motivate SFT, and inferior choices among valid answers motivate preference learning.
>
> **Where you are:** The transformer chapters explained next-token prediction. Part 1 changes **what behavior the examples teach**; Part 2 changes **where the update is stored**; Part 3 asks **what the evidence supports**.
>
> **Notation:** $x$ is a prompt or manuscript context; $y$ is a target response; $y^+$ and $y^-$ are editor-chosen and rejected responses; $\pi_\theta$ is the trainable model; $\pi_{\mathrm{ref}}$ is the frozen SFT reference used by DPO.

## Riverside's Brief

Riverside House has seven unpublished novels, about 197 chapters and 619,000 words. Manuscript text cannot leave the building, so every teaching run uses `Qwen/Qwen2.5-0.5B-Instruct` locally.

The immediate job is an editing assistant that can continue Riverside prose, obey a bounded request, and favor the response an editor would keep. Those are distinct behaviors, so each receives its own training signal and acceptance probe.

| Part | Question |
| --- | --- |
| 1 - this notebook | What training experience addresses the observed failure? |
| [2 - parameter strategy](02-llm-finetuning-parameter-techniques.ipynb) | How much model state must move, and what remains resident? |
| [3 - comparison and decision](03-llm-finetuning-comparison-and-decision.ipynb) | What does each result prove, and what can ship? |

The runnable checkpoints are not one mandatory ancestry chain: continued pretraining and SFT start from separate base models; DPO continues from the SFT adapter.

## Corpus and Setup

The committed `content/` directory contains the private teaching corpus. Run `setup.ps1`, select the `llm-tuning` kernel, and execute from a clean kernel. Checkpoints are written under `./checkpoints/` for Parts 2 and 3.

> **Boundary:** fine-tuning changes persistent behavior. A later retrieval chapter supplies current, citable corpus facts.

## Fine-Tuning Roadmap: Start Here

This map stays with the three-notebook arc. Read it vertically: Part 1 teaches **what behavior to learn**, Part 2 changes **how many parameters learn it**, and Part 3 decides **which evidence matters for each workload**.

```mermaid
flowchart TD
    Start["Starting point<br/>Base Qwen2.5: fluent, but domain-blind"]

    subgraph Data["Part 1 - Current notebook: choose the learning objective"]
        direction TB
        C1["[Now] Concept 1<br/>Continued pretraining<br/>Teach catalog language and style"]
        C2["[Next] Concept 2<br/>SFT<br/>Teach instruction following"]
        C3["[Next] Concept 3<br/>DPO<br/>Teach editor preference"]
        C1 --> C2 --> C3
    end

    subgraph Params["Part 2 - Next notebook: choose the parameter strategy"]
        direction TB
        C4["Concept 4<br/>Full fine-tuning"]
        C5["Concept 5<br/>Partial freezing"]
        C6["Concept 6<br/>LoRA"]
        C7["Concept 7<br/>QLoRA + quantization"]
        C4 --> C5 --> C6 --> C7
    end

    Start --> C1
    C3 --> Saved["Part 1 checkpoint<br/>Three capability-focused artifacts"]
    Saved --> C4
    C7 --> Compare["Part 3<br/>Compare evidence and choose per workload"]
```

> The arrows are a learning sequence, not literal model ancestry. In the runnable examples, continued pretraining and SFT start from separate base models; DPO continues from the SFT adapter. The roadmap tracks questions answered, while checkpoint tables track the actual artifacts.

## Prerequisite Bridge: From Encoder-Decoder Attention to a Decoder-Only Assistant

The transformer foundations introduced three useful shapes: an **encoder** reads an entire input, a **decoder** predicts the next token while respecting a causal mask, and an **encoder-decoder** model lets a decoder attend to an encoded source through cross-attention. Riverside's assistant uses the decoder-only choice: at each turn, the user's instruction, any supplied scene, and the completion form one growing token sequence; causal self-attention lets each new token use everything to its left without seeing its own future.

| Foundation                | Role in this chapter                                              | Why Riverside needs it                                                                               |
| ------------------------- | ----------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| Causal decoder            | `Qwen/Qwen2.5-0.5B-Instruct` predicts the next token      | It can continue prose and answer prompts from one left-to-right context                              |
| Training objective        | Labels say which next tokens should become more likely            | Continued pretraining, SFT, and DPO each change what Riverside teaches the same decoder              |
| Encoder / retrieval later | Encodes a query and passages for matching                         | It finds current, citable manuscript evidence instead of asking the generator to remember every fact |

So this notebook changes **how a decoder-only model behaves**. It does not turn the model into a dependable catalog lookup system: that next requirement leads to hybrid retrieval after the fine-tuning decision.


## Three Failures, Three Training Signals

![Three fine-tuning data objectives: continued pretraining, supervised fine-tuning, and direct preference optimization](images/data-objectives-pipeline.png)

| Observed failure | Training experience | Technique | Success looks like |
| --- | --- | --- | --- |
| Catalog prose is generic or inconsistent | Predict the next token in raw Riverside text | **Continued pretraining** | Riverside continuations become less generic |
| The model continues a request instead of obeying it | Pair requests with desired responses | **SFT** | Task, format, and stopping rules are followed |
| Several responses are valid but not equally useful | Compare chosen and rejected responses | **DPO** | Editor-preferred responses gain ground |

The sequence is diagnostic, not compulsory. Stop when the required behavior passes its probe; do not add another objective merely because it exists.

> **Implementation preview:** the objective and parameter strategy are separate choices. This notebook uses full fine-tuning for the continued-pretraining demonstration, then small LoRA adapters for SFT and DPO so the runs fit local hardware. Treat LoRA here as a small trainable correction attached to a frozen base; Part 2 opens that black box and compares it with full and partial fine-tuning.

## Learning Route

1. Establish the unchanged base model and three acceptance probes.
2. Let catalog-fluency failure create the need for continued pretraining.
3. Let instruction non-compliance create the need for SFT and prompt masking.
4. Let competing valid answers create the need for preference data and DPO.
5. Compare the resulting checkpoints as behavior demonstrations, not a leaderboard.
6. Continue to Part 2 for parameter cost and Part 3 for workload evidence.

**Optional depth:** the token-level training-step microscope and production orchestration sections are references. Skip them on a first practical pass; return when debugging labels, gradients, or job boundaries.

In [ ]:
from pathlib import Path

# Resolve the notebook's own directory (content/ lives next to this notebook). VS Code's Jupyter
# kernels run with cwd = workspace root, not the notebook's folder, so __vsc_ipynb_file__ (which
# VS Code injects) is the reliable way to find it; __file__ covers plain .py execution.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the corpus at {CONTENT_DIR}. Open and run this notebook from its own "
        "location in the repo (learning/genai/04-llm/) so its content/ folder resolves correctly."
    )

print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel directory mappings (keys are shorthand aliases, values are actual directory names)
NOVELS = {
    "scifi": "the-weight-of-distant-light",  # 40 chapters
    "fantasy": "the-tidebound-accord",  # 33 chapters
    "mystery": "the-cartographers-cipher",  # 21 chapters
    "historical": "the-silk-merchants-daughter",  # 23 chapters
    "cyberpunk": "neural-drift",  # 24 chapters
    "horror": "the-hollow-beneath",  # 28 chapters
    "literary": "the-weight-of-tides",  # 28 chapters
    "everglades": "the-everglades-cipher",  # 28 chapters
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from selected novels, or the complete corpus when novels is None.

    Args:
        novels: Novel aliases to load, or None for every directory in NOVELS.
        max_chapters: Maximum chapters per novel, or None for every chapter.
        min_len: Skip paragraphs shorter than this many characters.

    Returns:
        List of paragraph strings from all requested novels.
    """
    if novels is None:
        novels = list(NOVELS.keys())

    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            print(f"Warning: unknown novel alias '{alias}', skipping")
            continue

        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            print(f"Warning: directory {novel_path} not found, skipping")
            continue

        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        if max_chapters is not None:
            chapter_files = chapter_files[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)

    return paragraphs


# Sample from 4 genres to show multi-genre paragraph diversity
sample_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery", "horror"], max_chapters=2
)
print(f"Loaded {len(sample_paragraphs)} sample paragraphs from 4 novels. First one:\n")
print(sample_paragraphs[20][:421], "...")

## Baseline: Let the Model Fail Before Naming a Technique

Riverside will reuse three probes after every training stage:

| Probe | Request | Failure signal |
| --- | --- | --- |
| Catalog fluency | Continue a passage containing Riverside-only names and relationships | Generic continuation or invented story facts |
| Instruction compliance | `Answer in one sentence and stop.` | Restates the request, rambles, or violates the format |
| Editor preference | Compare two valid answers to the same request | No consistent reason to favor the concise, useful answer |

Start with the catalog-fluency probe. The base model has never seen Riverside's manuscripts, so a fluent answer is not evidence of knowledge. It can only guess from names in the prompt.

That gives us the first failure to fix:

> The model knows how English works, but Riverside language is still surprising to it.

---

### Setting Up the Shared Baseline

All three stages use the same base checkpoint and fixed prompts. Keeping an untouched `base_model` gives every later comparison a real before state rather than a remembered sample.

### Pin a CPU-Friendly Base Model

Every comparison needs one unchanged starting point. Riverside uses `Qwen/Qwen2.5-0.5B-Instruct`, an instruction-tuned Qwen2 causal decoder with 494,032,768 parameters, 24 decoder blocks, and hidden size 896.

Comparing different models and their suitability on a GPU-enabled device is outside the scope of this notebook. Qwen2.5-0.5B is compact by modern LLM standards while retaining a credible general-language baseline. Changing `MODEL_NAME` invalidates checkpoints from another architecture.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
DEMO_TRAIN_STEPS = 60
DEMO_DPO_STEPS = 60

### Choosing a Device: GPU if Available, CPU Otherwise

`device` tells PyTorch where tensors and model weights should physically live. We need this because
every tensor operation in this notebook (forward pass, backward pass, `.generate()`) has to run on
the same device as the weights, or PyTorch raises a device-mismatch error.

`torch.cuda.is_available()` checks for a usable NVIDIA GPU + CUDA driver; if none is found, we fall
back to `"cpu"` so the notebook still runs end-to-end (just slower) on a laptop with no dedicated
GPU -- Riverside's actual situation.


> **PyTorch → Keras:** `torch.cuda.is_available()` — checks whether a CUDA-capable GPU is visible to PyTorch and returns a bool; the result picks the `device` string (`"cuda"` or `"cpu"`) that every tensor and model call below is pinned to via `.to(device)`. **Keras/TF equivalent:** `tf.config.list_physical_devices('GPU')` — TensorFlow auto-places ops on any visible GPU without needing an explicit device string threaded through the code, so most Keras code skips this check entirely; `tf.device(...)` exists for the rare case you want to force placement.

In [ ]:
device = (
    "cuda" if torch.cuda.is_available() else "cpu"
)  # detects whether a GPU is available
print(f"Using device: {device}")

### Loading the Tokenizer and Instruction Format

The checkpoint uses Qwen2.5's 151,936-token vocabulary and built-in chat template. Its pad token is `<|endoftext|>`, while `<|im_end|>` terminates chat messages. Whitespace and punctuation can change token boundaries, so the following cells inspect actual IDs rather than assuming word-level tokens.

SFT, DPO, and instruction evaluation all use the tokenizer's native system/user/assistant serialization:

```text
<|im_start|>system
...
<|im_end|>
<|im_start|>user
...
<|im_end|>
<|im_start|>assistant
...
<|im_end|>
```

For batching, padded labels are masked with `-100`, so padding contributes no training loss.


> **PyTorch → Keras:** `AutoTokenizer.from_pretrained(MODEL_NAME)` loads the same checkpoint-matched
tokenizer for either framework. Tokenization and explicit instruction-text serialization are framework-agnostic; the split
between PyTorch and TensorFlow begins only when the resulting arrays are converted to framework tensors.

In [ ]:
# Qwen2.5 provides the chat template used consistently by SFT, DPO, and evaluation.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_instruction(instruction, response=None):
    """Render one native Qwen chat contract for SFT, DPO, and evaluation."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction.strip()},
    ]
    if response is None:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    messages.append({"role": "assistant", "content": response.strip()})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


print(f"Tokenizer vocabulary size: {len(tokenizer):,}")
print(f"Pad token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(render_instruction("Continue this sentence: The signal arrived")[:240])


### Seeing the Vocabulary in Action

Byte-level BPE can represent arbitrary text, but common spans receive compact tokens while rare names or
unusual Unicode sequences split into several pieces. Whitespace is context: tokenizing `"signal"` and
`" signal"` can produce different IDs because a space may be merged with neighboring bytes.

The code below prints IDs, raw tokenizer tokens, and decoded pieces for each example. Decoding each ID is
the portable way to make spaces and newlines visible; raw token strings are implementation details and
should not be treated as a universal notation.


> **PyTorch → Keras:** `tokenizer.encode(word)` / `tokenizer.convert_ids_to_tokens(ids)` — converts raw text to integer token IDs (and back to readable BPE-piece strings) using the framework-agnostic tokenizer loaded above; no tensors are created yet, just plain Python lists. **Keras/TF equivalent:** identical call — `AutoTokenizer` isn't PyTorch- or TF-specific, so a Keras/TF version of this notebook would use this exact same code; only the downstream model call (`TFAutoModelForCausalLM` vs. `AutoModelForCausalLM`) would differ.

In [ ]:
# One example each of a noun, proper noun, verb, and adjective -- all pulled from Riverside's own
# sci-fi opening line, so these are words this notebook already leans on elsewhere.
example_words = {
    "noun": "signal",
    "proper noun": "Aria",
    "verb": "stared",
    "adjective": "distant",
}

for part_of_speech, word in example_words.items():
    ids_alone = tokenizer.encode(word)  # tokenize the word standalone (no leading space)
    ids_mid_sentence = tokenizer.encode(" " + word)  # tokenize as it would appear mid-sentence
    print(f"{part_of_speech.upper()}: {word!r}")
    print(
        f"  as the first word of a text  : ids={ids_alone}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_alone)}"
    )
    print(
        f"  mid-sentence (' {word}')".ljust(31) + f": ids={ids_mid_sentence}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_mid_sentence)}"
    )
    print()

print(
    "'\u0120' at the start of a token marks a leading space -- it's why the same word can tokenize "
    "differently depending on where it appears in a sentence."
)


> **You may wonder:** since `Aria` splits into two tokens and appears constantly in this corpus, why
> not just train the tokenizer on Riverside's own text and merge it into a dedicated token? Two
> reasons this is out of scope for fine-tuning: extending the vocabulary adds a new, untrained row
> to the embedding matrix and output head, and filling that row in with a meaningful representation
> is itself a training problem, not something fine-tuning does for free. And splitting `Aria` into
> two tokens doesn't stop the model from learning what it means -- it can still learn to associate
> that two-token pattern with everything fine-tuning teaches it about her; it just costs two sequence
> positions instead of one, a small efficiency tax, not a correctness problem. The actual gap the
> rest of this notebook closes is that the model has never seen who Aria Voss is, not how her name
> happens to be tokenized.



> **A related question:** what happens with a word the tokenizer has rarely encountered? Byte-level
> BPE does not need an unknown-word vocabulary entry: when no longer merge matches, it falls back to smaller
> byte-derived pieces. An uncommon name such as `Itzpapalotl` therefore remains representable, although it
> usually consumes more tokens than a frequent word. Fine-tuning can improve how the model uses that sequence,
> but it does not add a new vocabulary row unless the tokenizer and embedding matrix are explicitly resized.


### Loading the Base Model

`base_model` is the actual pretrained neural network -- a checkpoint-defined number of real weights downloaded from the
Hugging Face hub, moved onto whichever device we resolved above via `.to(device)`. This untouched
checkpoint is the "before" every fine-tuning technique in this notebook is compared against.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` — downloads the weights selected by `MODEL_NAME` into a PyTorch `nn.Module` and moves every parameter tensor onto `device` (CPU or GPU) in place. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` — loads the same checkpoint into a `tf.keras.Model` instead; TensorFlow doesn't need an explicit `.to(device)` call since ops are placed on available devices automatically (or via a `tf.device(...)` context).

In [ ]:
# Download the lightweight base model and keep its architecture facts runtime-derived.
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
decoder_blocks = base_model.model.layers
n_blocks = len(decoder_blocks)
hidden_size = base_model.config.hidden_size
print(
    f"Loaded {MODEL_NAME}: {model_parameter_count:,} parameters, "
    f"{n_blocks} decoder blocks, hidden size {hidden_size}."
)

### A Fixed Test Prompt for Before/After Comparisons

`PROMPT` is the one fixed test sentence reused throughout the notebook so "before" vs. "after"
fine-tuning comparisons are always apples-to-apples. It's pulled straight from the sci-fi corpus so
a model that has actually absorbed the catalog has a real chance of continuing it in-world.


In [ ]:
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus

### A Reusable `generate()` Helper

Hugging Face returns the prompt and completion in one token sequence. The helper records the prompt length, slices `output[prompt_length:]`, and decodes only the new tokens so every comparison shows the model's actual continuation.

It also calls `.strip()` because the first generated token may carry leading whitespace. All later candidates use this same helper, so output formatting cannot masquerade as a model difference.

> **PyTorch → Keras:** `model.eval()` / `torch.no_grad()` / `model.generate()` — `.eval()` switches dropout/batchnorm-style layers to inference mode, `torch.no_grad()` disables gradient tracking to save memory during inference, and `.generate()` runs HuggingFace's autoregressive sampling loop (nucleus sampling here via `top_p`/`temperature`). **Keras/TF equivalent:** `TFAutoModelForCausalLM.generate()` — the same HuggingFace `.generate()` API exists on TF models with identical sampling arguments; TF's analog of "eval mode" is passing `training=False` (implicit inside `.generate()`), and there's no separate "no_grad" context since calling a `tf.keras.Model` outside a `GradientTape` block already skips gradient recording.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base model, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.

    Notes
    -----
    A real example from this notebook's own `PROMPT` (15 tokens) makes both
    lines concrete. Asking for `max_new_tokens=15` returns `out` with shape
    `(1, 30)` -- the 15 prompt tokens plus 15 new ones, concatenated. Decoding
    all 30 without slicing prints the prompt right back before the answer:

        'Aria Voss stared at the signal counting itself out in prime numbers
         and began to ponder the question, what was it that she had to do?'

    `out[0][prompt_len:]` (`prompt_len = 15` here) drops the first 15 tokens so
    only the new continuation gets decoded. But decoding *just* those 15 new
    tokens gives:

        ' began to ponder the question, what was it that she had to do?'

    -- note the stray leading space: the decoded continuation can begin with whitespace carried by its first
    token, so the raw decoded string may start with a space. `.strip()`
    removes it, along with any trailing whitespace/newlines near the end.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(
        device
    )  # use the same tokenizer to tokenize the prompt and convert it to tensor
    prompt_len = inputs["input_ids"].shape[1]  # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # stochastic → varied output
            top_p=0.9,  # nucleus sampling: top 90% mass
            temperature=0.8,  # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )

    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return (
        completion
        if completion
        else "[model stopped immediately — sampled EOS as first token]"
    )


print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")
print(f"Completion: {generate(base_model, PROMPT, 20)}")

### Code Walkthrough: Shared Setup

**1. Padding fallback**

The GPT-2 tokenizer has no dedicated pad token. Reusing EOS is safe here because every padding label is replaced with `-100` before loss is computed.

**2. Runtime-derived architecture**

`AutoModelForCausalLM.from_pretrained(MODEL_NAME)` loads the checkpoint. Parameter count, decoder-block count, and hidden width come from the loaded object rather than hard-coded prose.

**3. Explicit instruction formatting**

`render_instruction()` uses the same `### Instruction` / `### Response` boundaries for SFT examples, DPO pairs, and instruction evaluation. Continued pretraining remains plain causal text.

## Test Prompts: Define Success Before Training

A fluent continuation is not enough; the base model is already fluent. A successful Riverside adaptation must use story-specific entities coherently without copying the prompt or inventing another world.

Use three probe families:

| Probe | What it checks | Failure signal |
| --- | --- | --- |
| Character and setting | Catalog-specific relationships | Generic roles, places, or invented lore |
| Genre continuation | Prose and narrative behavior | Fluent text in the wrong voice or genre |
| Cross-novel vocabulary | Breadth beyond one manuscript | Improvement only on memorized phrases |

For example, after `Aria Voss checked the Meridian's Promise status panel and`, an adapted continuation should remain aboard Riverside's ship and use established relationships; mentioning the name alone does not count.

The executable `TEST_PROMPTS` dictionary in the next cell is the source of truth for the full multi-genre fixture set. Keeping the prompts in code avoids maintaining the same catalog twice.

In [ ]:
# Automated test runner: compare baseline vs fine-tuned on corpus-specific prompts
TEST_PROMPTS = {

    # Sci-fi — The Weight of Distant Light
    "scifi_character": "Aria Voss checked the Meridian's Promise status panel and",
    "scifi_keeper": "The Keeper's consciousness flickered through node seventeen as",

    # Fantasy — The Tidebound Accord
    "fantasy_magic": "Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as",
    "fantasy_hollow_king": "The Hollow King's followers, called the Hollowed, began to gather when",

    # Mystery — The Cartographer's Cipher
    "mystery_conspiracy": "The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—",
    "mystery_elena": "Elena Voss studied the 1879 survey map and realized the Ashmont Trust",

    # Historical — The Silk Merchant's Daughter
    "historical_setting": "Wei Lian's jade phoenix pendant caught the morning light in Chang'an as",
    "historical_silk_road": "The delegation crossed the Taklamakan desert and Wei Lian noted in her ledger",

    # Cyberpunk — Neural Drift
    "cyberpunk_tech": "Kai Chen adjusted the neurorig and prepared to extract the memory backup from",
    "cyberpunk_project": "In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift",

    # Horror — The Hollow Beneath
    "horror_atmosphere": "Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor",
    "horror_chambers": "The tenth chamber of the Hollow pulsed with a light that had no source, and Eleanor",

    # Literary — The Weight of Tides
    "literary_marine": "Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about",
    "literary_contact": "The Observer surfaced near Whitehead Island and Claire understood for the first time that",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run all test prompts and return results dict for comparison."""
    results = {}
    prompts = {}
    for key, prompt in test_prompts.items():
        prompts[key] = prompt
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)  # this model's continuation for each prompt
    return results, prompts


# Run baseline tests (will show generic, off-corpus continuations)
print(
    "=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===\n"
)
baseline_results, prompts = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(prompts[key] + " .... " + output[:200] + "...\n")


Notice the output has no awareness of Aria Voss (from _The Weight of Distant Light_), the _Meridian's
Promise_, the Lantern, or any of the other characters/worlds across the seven novels -- it is fluent
English but a generic, unrelated continuation. This is exactly the gap fine-tuning closes.

Per the "What 'success' actually looks like" example above, a model that had genuinely absorbed this
corpus would instead keep Aria aboard the Meridian's Promise, in her actual role, referencing the
Under-Hold or the Lantern instead of inventing an unrelated ship and crew. This notebook doesn't
re-run `test_corpus_knowledge()` on a fine-tuned checkpoint (fine-tuning a fresh model per test prompt
would multiply the compute cost of every section below), but the Ablation Study near the end trains
and compares checkpoints on a closely related Aria Voss / Meridian's Promise prompt, so you can see
the real before/after side by side.


### Before the Mechanics: What One Training Step Is Trying to Change

Use the notebook's Aria Voss sentence as the local walking example. Before adaptation, the model sees a prefix such as `Aria Voss stared at the signal` and spreads its expectation across many possible next tokens. The manuscript then reveals what actually came next.

One training step asks a simple question:

> After reading this prefix, how much expectation did the model place on the manuscript's actual next token?

If the actual token received little probability, the step produces a larger correction. If it was already expected, the correction is smaller. Repeating this across every position lets one paragraph supply many small lessons at once.

The detailed walkthrough below explains how the implementation performs that loop efficiently:

1. represent the text as token IDs;
2. prevent each position from seeing future tokens;
3. compare each position's prediction with the actual next token;
4. ignore padding that contains no lesson;
5. adjust trainable weights so the observed continuation becomes a little less surprising.

The arrays, masks, loss, gradients, and optimizer are machinery for this idea. Keep returning to the same question: **what did the model expect next, what actually came next, and how should that expectation move?**

### Optional Depth: Inspect One Training Step

The practical fine-tuning path needs one mental model: the tokenizer produces causal-LM examples, the model assigns probability to each actual next token, the loss measures surprise, and backpropagation adjusts whichever parameters are trainable.

The cells below open that loop position by position using a Riverside paragraph. They cover token shifts, causal masking, cross-entropy, gradients, and optimizer updates with animations.

Skip to **The Fine-Tuning Journey** if those mechanics are already familiar. For FDE and evaluation work, you need to interpret loss and verify masking; you do not need to memorize every intermediate tensor or animation.

This is the first section that actually plots anything, so this is where we load the visualization
stack -- `matplotlib`/`seaborn` for the charts, `numpy` for the array math behind them. Every later
section that visualizes training internals reuses these same imports.


In [ ]:
# Visualization imports for intuition building
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
from IPython.display import display, HTML
import warnings

warnings.filterwarnings("ignore")  # suppress noisy library warnings for cleaner notebook output
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})  # sharper plots, readable default font size
sns.set_theme(style="whitegrid", palette="muted")  # consistent seaborn styling across the notebook

print("Visualization libraries loaded.")


### Running Steps 1-2 for Real

The walkthrough above described this in the abstract -- let's actually run it, one step at a time, on
`Qwen/Qwen2.5-0.5B-Instruct` and our example paragraph. First, Steps 1-2: tokenize the sentence into fixed-length
tensors, then build `labels` (a clone of `input_ids`, with padding positions masked to `-100`) exactly
the way described above.


> **PyTorch → Keras:** `input_ids.clone()` / `.to(device)` / boolean-index assignment (`labels[attention_mask == 0] = -100`) — `.clone()` makes an independent copy of a tensor so mutating `labels` won't affect `input_ids`, `.to(device)` moves it to CPU/GPU, and the boolean mask assignment overwrites padding positions in place. **Keras/TF equivalent:** `tf.identity(input_ids)` for the copy — TF tensors are immutable, so masking instead builds a *new* tensor via `tf.where(attention_mask == 0, -100, input_ids)` rather than an in-place assignment; device placement is again automatic instead of an explicit `.to(device)` call.

In [ ]:
import torch.nn.functional as F

example_text = (
    "Aria Voss stared at the signal counting itself out in prime numbers and felt the "
    "weight of two centuries press against her ribs."
)  # same sentence used in the Step 1-2 walkthrough above

# Step 1: tokenize
enc = tokenizer(
    example_text,
    truncation=True,
    max_length=128,
    padding="max_length",
    return_tensors="pt",
)  # returns input_ids + attention_mask, both padded/truncated to 128
input_ids = enc["input_ids"].to(device)  # real token ids, followed by pad-token IDs
attention_mask = enc["attention_mask"].to(device)  # 1 = real token, 0 = padding
real_len = int(attention_mask.sum().item())  # number of real (non-padding) tokens

# Step 2: build labels -- identical to input_ids, padding positions masked to -100
labels = input_ids.clone()  # not shifted -- labels[i] currently equals input_ids[i]
labels[attention_mask == 0] = -100  # wherever attention_mask is 0 (padding), set that label to -100 so the loss skips it

print(
    f"Tokenized to {input_ids.shape[1]} total positions: {real_len} real tokens + "
    f"{input_ids.shape[1] - real_len} padding tokens"
)
print(f"first 8 input_ids : {input_ids[0, :8].tolist()}")
print(
    f"first 8 labels    : {labels[0, :8].tolist()}  <- identical to input_ids (not shifted)"
)
print(
    f"last 8 labels     : {labels[0, -8:].tolist()}  <- all -100 (padding, ignored by the loss)"
)


### Visual Guide: Masking and the Shift, Panel by Panel

Steps 1-2 above are dense in prose -- the figure right below turns them into a picture, built from the
_real_ `input_ids`, `attention_mask`, and `labels` just computed for our example paragraph (nothing
here is a schematic with made-up numbers). Four panels, each isolating one piece of the mechanism:

- **Panel A** -- what `input_ids` actually holds: the real token stream, decoded back into text.
- **Panel B** -- what happens right at the real/padding boundary (position `real_len`): real tokens
  keep their own id as the label; padding positions get overwritten to `-100`.
- **Panel C** -- the shift itself: position `i`'s prediction is graded against the label sitting one
  slot ahead, `label[i+1]` -- the exact mechanism the write-up above described in words only.
- **Panel D** -- the payoff of `-100`: which positions the loss actually counts, and which it silently
  skips via `ignore_index=-100`.


In [ ]:
# Visual guide: masking and the shift, panel by panel -- built from the real tokenizer/model
# output above (input_ids, labels, attention_mask, real_len), not a fabricated schematic.
from matplotlib.patches import Patch


def _clean_tok(token_id):
    """Decode one token ID and make whitespace visible in a plot."""
    piece = tokenizer.decode([token_id], skip_special_tokens=False)
    return piece.replace(" ", "·").replace("\n", "\\n")


decoded_tokens = [_clean_tok(token_id) for token_id in input_ids[0].tolist()]

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle(
    "Masking and Prediction, Panel by Panel -- real tokens from MODEL_NAME's tokenizer",
    fontsize=13,
    fontweight="bold",
)

# Panel A: the real token stream -- what input_ids actually holds
ax_a = axes[0, 0]
n_show_a = 8
for pos in range(n_show_a):
    ax_a.add_patch(
        Rectangle((pos, 0), 0.9, 1, facecolor="lightblue", edgecolor="black")
    )
    ax_a.text(
        pos + 0.45,
        0.5,
        decoded_tokens[pos],
        ha="center",
        va="center",
        fontsize=8,
    )
    ax_a.text(
        pos + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray"
    )
ax_a.set_xlim(-0.2, n_show_a + 0.2)
ax_a.set_ylim(-0.6, 1.3)
ax_a.axis("off")
ax_a.set_title(
    "Panel A: input_ids -- the real token stream (position below each box)",
    fontsize=10,
    fontweight="bold",
)
ax_a.legend(
    handles=[Patch(facecolor="lightblue", edgecolor="black", label="Real token")],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18),
    fontsize=7,
)

# Panel B: the real/padding boundary -- where labels actually get masked to -100
ax_b = axes[0, 1]
start_b = max(0, real_len - 5)
end_b = min(input_ids.shape[1], real_len + 4)
window_b = list(range(start_b, end_b))
boundary_j = real_len - start_b
for j, pos in enumerate(window_b):
    is_real = pos < real_len
    color = "mediumseagreen" if is_real else "lightgray"
    label_text = decoded_tokens[pos] if is_real else "-100"
    ax_b.add_patch(Rectangle((j, 0), 0.9, 1, facecolor=color, edgecolor="black"))
    ax_b.text(j + 0.45, 0.5, label_text, ha="center", va="center", fontsize=8)
    ax_b.text(
        j + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray"
    )
boundary_line_b = ax_b.axvline(
    boundary_j,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Real/padding boundary",
)
ax_b.set_xlim(-0.2, len(window_b) + 0.2)
ax_b.set_ylim(-0.6, 1.3)
ax_b.axis("off")
ax_b.set_title(
    f"Panel B: labels right at the boundary (real_len={real_len})",
    fontsize=10,
    fontweight="bold",
)
ax_b.legend(
    handles=[
        Patch(
            facecolor="mediumseagreen",
            edgecolor="black",
            label="Real token (label = same token)",
        ),
        Patch(facecolor="lightgray", edgecolor="black", label="Padding (label = -100)"),
        boundary_line_b,
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.3),
    fontsize=7,
)

# Panel C: the shift itself -- prediction at position i graded against label[i+1]
ax_c = axes[1, 0]
n_show_c = 6
for pos in range(n_show_c):
    ax_c.add_patch(
        Rectangle((pos, 0.7), 0.9, 1, facecolor="#ffd9a0", edgecolor="black")
    )
    ax_c.text(
        pos + 0.45,
        1.2,
        decoded_tokens[pos],
        ha="center",
        va="center",
        fontsize=8,
    )
    ax_c.text(
        pos + 0.45,
        1.85,
        f"logits[{pos}]",
        ha="center",
        va="center",
        fontsize=6.5,
        color="gray",
    )
    ax_c.add_patch(
        Rectangle((pos, -1.2), 0.9, 1, facecolor="lightblue", edgecolor="black")
    )
    ax_c.text(
        pos + 0.45,
        -0.7,
        decoded_tokens[pos + 1],
        ha="center",
        va="center",
        fontsize=8,
    )
    ax_c.text(
        pos + 0.45,
        -1.45,
        f"label[{pos + 1}]",
        ha="center",
        va="center",
        fontsize=6.5,
        color="gray",
    )
    ax_c.annotate(
        "",
        xy=(pos + 0.45, -0.15),
        xytext=(pos + 0.45, 0.65),
        arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5),
    )
ax_c.set_xlim(-0.2, n_show_c + 0.2)
ax_c.set_ylim(-1.7, 2.2)
ax_c.axis("off")
ax_c.set_title(
    "Panel C: the shift -- position i's prediction is graded against label[i+1]",
    fontsize=10,
    fontweight="bold",
)
ax_c.legend(
    handles=[
        Patch(
            facecolor="#ffd9a0",
            edgecolor="black",
            label="Token at position i (what the model has read)",
        ),
        Patch(
            facecolor="lightblue",
            edgecolor="black",
            label="label[i+1] -- what it's graded against",
        ),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.32),
    fontsize=7,
)

# Panel D: the payoff of -100 -- which positions the loss actually counts
ax_d = axes[1, 1]
for j, pos in enumerate(window_b):
    is_real = pos < real_len
    color = "coral" if is_real else "whitesmoke"
    ax_d.add_patch(Rectangle((j, 0), 0.9, 1, facecolor=color, edgecolor="black"))
    if not is_real:
        ax_d.text(
            j + 0.45,
            0.5,
            "skipped",
            ha="center",
            va="center",
            fontsize=7,
            color="gray",
            fontweight="bold",
        )
    ax_d.text(
        j + 0.45, -0.35, str(pos), ha="center", va="center", fontsize=7, color="gray"
    )
boundary_line_d = ax_d.axvline(
    boundary_j,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Real/padding boundary",
)
ax_d.set_xlim(-0.2, len(window_b) + 0.2)
ax_d.set_ylim(-0.6, 1.3)
ax_d.axis("off")
ax_d.set_title(
    "Panel D: ignore_index=-100 in action -- padding contributes zero loss",
    fontsize=10,
    fontweight="bold",
)
ax_d.legend(
    handles=[
        Patch(facecolor="coral", edgecolor="black", label="Counted in the loss"),
        Patch(
            facecolor="whitesmoke",
            edgecolor="black",
            label="Skipped (ignore_index=-100)",
        ),
        boundary_line_d,
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.3),
    fontsize=7,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(
    f"Real/padding boundary for our example paragraph: position {real_len} out of {input_ids.shape[1]}."
)
print(
    "Panels A-D all use the real tokenizer/model output computed above -- nothing here is fabricated."
)

### Watching the Shift Happen, Frame by Frame

Panel C above shows the shift as a static snapshot of six positions at once -- useful, but it still
asks you to hold "position `i` pairs with label `i+1`" in your head across six boxes simultaneously.
The animation below turns that same idea into a sequence: **one frame per token position**, revealed
one at a time, so the pairing rule shows up as a repeating motion instead of a paragraph of prose.

Watch specifically for two things as the frames advance:

1. The red arrow always points from the top row (position `i`, what the model has just read) down to
   the bottom row **one slot to the right** (label `i+1`) -- never straight down. That one-slot offset
   _is_ the entire "shift" this section has been building up to.
2. The `labels` array itself never rearranges -- every bottom-row token is exactly the same token
   that already sits in `input_ids` at that position. The animation only ever reveals a new _pairing_
   each frame, never a new array.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# One frame per real token position -- reveals the top/bottom pairing one step at a time instead
# of showing all six pairs at once like the static Panel C above.
n_frames_shift = min(10, real_len - 1)

fig_shift, ax_shift = plt.subplots(figsize=(11, 4.5))


def _update_shift(frame):
    ax_shift.clear()
    ax_shift.set_xlim(-0.3, n_frames_shift + 0.3)
    ax_shift.set_ylim(-2.3, 2.3)
    ax_shift.axis("off")

    for pos in range(n_frames_shift):

        # Top row: token AT position `pos` -- what the model has read up through this step
        if pos < frame:
            top_color = "#c9e6c9"  # already graded this pair -- "done"
        elif pos == frame:
            top_color = "#ffb347"  # the pair being graded THIS frame
        else:
            top_color = "whitesmoke"  # not reached yet
        ax_shift.add_patch(
            Rectangle((pos, 0.7), 0.9, 1, facecolor=top_color, edgecolor="black")
        )
        ax_shift.text(
            pos + 0.45,
            1.2,
            decoded_tokens[pos],
            ha="center",
            va="center",
            fontsize=9,
        )
        ax_shift.text(
            pos + 0.45,
            1.85,
            f"pos {pos}",
            ha="center",
            va="center",
            fontsize=7,
            color="gray",
        )

        # Bottom row: label[pos + 1] -- only revealed once its pair has actually been reached
        if pos <= frame:
            bottom_color = "#c9e6c9" if pos < frame else "#8ecae6"
            ax_shift.add_patch(
                Rectangle(
                    (pos, -1.7), 0.9, 1, facecolor=bottom_color, edgecolor="black"
                )
            )
            ax_shift.text(
                pos + 0.45,
                -1.2,
                decoded_tokens[pos + 1],
                ha="center",
                va="center",
                fontsize=9,
            )
            ax_shift.text(
                pos + 0.45,
                -1.95,
                f"label[{pos + 1}]",
                ha="center",
                va="center",
                fontsize=7,
                color="gray",
            )
            ax_shift.annotate(
                "",
                xy=(pos + 0.45, -0.65),
                xytext=(pos + 0.45, 0.65),
                arrowprops=dict(
                    arrowstyle="->",
                    color="darkred" if pos == frame else "#9aa5b1",
                    lw=2.2 if pos == frame else 1,
                ),
            )
        else:
            ax_shift.add_patch(
                Rectangle(
                    (pos, -1.7), 0.9, 1, facecolor="whitesmoke", edgecolor="black"
                )
            )
            ax_shift.text(
                pos + 0.45,
                -1.2,
                "?",
                ha="center",
                va="center",
                fontsize=9,
                color="lightgray",
            )

    current_tok = decoded_tokens[frame]
    label_tok = decoded_tokens[frame + 1]
    ax_shift.set_title(
        f"Position {frame}: model has read through '{current_tok}' \u2192 prediction here is graded "
        f"against label[{frame + 1}] = '{label_tok}'\n(same labels array throughout -- only the "
        f"pairing shown by the arrow shifts)",
        fontsize=10,
        fontweight="bold",
    )
    return []


anim_shift = FuncAnimation(
    fig_shift, _update_shift, frames=n_frames_shift, interval=900, blit=False
)
plt.close(fig_shift)  # prevent a duplicate static frame from also rendering

print(
    "Animation: one frame per token position. Top row = what the model has read so far; bottom row = "
    "the label it's graded against for that position's prediction. The red arrow always points from "
    "position i (top) down to label[i+1] (bottom, one slot over) -- that one-slot offset is the whole "
    "'shift'. Green = already-graded pairs; gray = not reached yet.\n"
)
display(HTML(anim_shift.to_jshtml(fps=2)))

### Steps 3-5: Forward Pass, Loss, and Backprop in One Call

Passing `labels=...` into `base_model(...)` makes HuggingFace do Steps 3 and 4 internally in a single
call: it runs the forward pass (producing `outputs.logits`), then shifts and compares logits against
labels the way described above, returning the averaged result as `outputs.loss`. `step_loss.backward()`
is Step 5 -- PyTorch's autograd walks backward through every operation that produced `step_loss` and
computes `∂Loss/∂W` for every parameter that needs a gradient, without us deriving any calculus by hand.
`base_model.train()` beforehand just tells dropout-style layers to behave in "training mode" for this
one pass; it's switched back to `.eval()` a couple of cells down, once this illustrative pass is done,
so nothing here leaks into the rest of the notebook.


> **PyTorch → Keras:** `model.train()` / `model.zero_grad()` / `model(..., labels=...)` / `loss.backward()` — `.train()` re-enables dropout for this illustrative pass, `.zero_grad()` clears stale gradients from any previous backward call, passing `labels=` makes the HuggingFace model compute cross-entropy internally and return it as `outputs.loss`, and `.backward()` triggers PyTorch autograd to populate `.grad` on every parameter. **Keras/TF equivalent:** `tf.GradientTape()` — Keras/TF has no separate "training mode" flag on the model itself (`training=True` is passed as a call argument instead) and no manual `.backward()`; you'd wrap the forward pass in `with tf.GradientTape() as tape:`, then call `tape.gradient(loss, model.trainable_variables)` to get the equivalent of populated `.grad` attributes.

In [ ]:
base_model.train()  # need gradients for this one illustrative pass; restored to eval() below
base_model.zero_grad()  # clear any stale .grad values before this pass (defensive habit -- none exist yet)

# Use eager attention for this illustrative pass so output_attentions exposes per-position weights.
# Restore the configured backend immediately afterward.
_prev_attn_impl = base_model.config._attn_implementation  # remember the current backend so it can be restored
base_model.set_attn_implementation("eager")  # eager computes attention explicitly, so it can expose per-position weights
outputs = base_model(
    input_ids=input_ids,  # the real + padding token ids from Step 1
    attention_mask=attention_mask,  # blocks every position from attending to padding tokens
    labels=labels,  # triggers HuggingFace's internal logits[:-1] vs labels[1:] loss calculation
    output_attentions=True,  # keeps the per-layer attention weights for the animation below
)
base_model.set_attn_implementation(_prev_attn_impl)  # restore the original (SDPA) attention backend
step_loss = outputs.loss  # the single cross-entropy value HuggingFace averaged over all real positions
step_loss.backward()  # compute gradients for every parameter via backprop

print(
    f"logits shape: {tuple(outputs.logits.shape)}  (one {base_model.config.vocab_size:,}-vocab prediction per position)"
)
print(f"average loss across all real positions: {step_loss.item():.3f}")


### Quick Revision: Parallel Causal Attention

Unlike an RNN, a causal transformer processes all token positions in parallel. The causal mask prevents each position from seeing future tokens, so one forward pass produces a next-token prediction -- and a training-loss contribution -- at every eligible position.

For the full treatment, revisit [Part 4 - Queries, Keys & Values](../02-transformers/transformers.ipynb#part-4---queries-keys-values) and [Part 5 - Multi-Head Attention](../02-transformers/transformers.ipynb#part-5---multi-head-attention). The animation below is only a quick visual recap, using real layer-0, head-0 attention weights from the forward pass above.

> **PyTorch → Keras:** `outputs.attentions[0][0, 0, :window_len, :window_len].detach().cpu().numpy()` — `.detach()` removes a tensor from the autograd graph so no gradient bookkeeping follows it, `.cpu()` moves it off the GPU if it was there, and `.numpy()` converts it to a plain NumPy array for matplotlib to plot. **Keras/TF equivalent:** `tensor.numpy()` — TF's eager-mode tensors expose `.numpy()` directly with no separate detach/cpu step needed, since tensors returned outside a `GradientTape` already carry no gradient history and `.numpy()` implicitly copies off-device if necessary.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Real attention weights (layer 0, head 0) for a 4-token window at the front of our example
# paragraph -- pulled straight from outputs.attentions returned by the Step 3 forward pass above,
# not fabricated. Row i = how much query position i attends to each key position j <= i.
window_len = 4
window_tokens = decoded_tokens[:window_len]
attn_window = (
    outputs.attentions[0][0, 0, :window_len, :window_len].detach().float().cpu().numpy()
)
vmax_attn = attn_window.max()

fig_attn, (ax_seq, ax_par) = plt.subplots(1, 2, figsize=(11, 4.6))


def _row_grid(up_to_row):
    """Rows 0..up_to_row filled with real attention weights (causal-masked); later rows blank."""
    grid = np.full((window_len, window_len), np.nan)
    for i in range(window_len):
        if i <= up_to_row:
            grid[i, : i + 1] = attn_window[i, : i + 1]
    return grid


def _draw_attn(ax, grid, title):
    ax.clear()
    ax.imshow(grid, cmap="viridis", vmin=0, vmax=vmax_attn, aspect="equal")
    ax.set_xticks(range(window_len))
    ax.set_yticks(range(window_len))
    ax.set_xticklabels(window_tokens, fontsize=8)
    ax.set_yticklabels(window_tokens, fontsize=8)
    ax.set_xlabel("key position j", fontsize=8)
    ax.set_ylabel("query position i", fontsize=8)
    ax.set_title(title, fontsize=9, fontweight="bold")
    for i in range(window_len):
        for j in range(window_len):
            if not np.isnan(grid[i, j]):
                ax.text(
                    j,
                    i,
                    f"{grid[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white" if grid[i, j] < vmax_attn * 0.6 else "black",
                )


n_frames_attn = window_len + 1  # one extra hold frame at the end


def _update_attn(frame):
    seq_row = min(frame, window_len - 1)
    _draw_attn(
        ax_seq,
        _row_grid(seq_row),
        f"If attention ran sequentially\n(row {seq_row} just 'arrived')",
    )
    _draw_attn(
        ax_par,
        _row_grid(window_len - 1),
        "What one forward pass actually returns\n(all rows already computed, one matmul)",
    )
    fig_attn.suptitle(
        f"Frame {frame + 1}/{n_frames_attn} -- real layer-0/head-0 attention, first {window_len} tokens",
        fontsize=10,
    )
    return []


anim_attn = FuncAnimation(
    fig_attn, _update_attn, frames=n_frames_attn, interval=900, blit=False
)
plt.close(fig_attn)  # prevent a duplicate static frame from also rendering

print(
    "Animation: the same real layer-0/head-0 attention weights on both sides. Left panel pretends "
    "attention arrives one row per step, like an RNN reading left to right. Right panel shows what "
    "base_model(...) actually returns -- every row already filled in after a single parallel matmul, "
    "because the causal mask (blocked/blank cells) already guarantees row i never depended on the "
    "rows below it.\n"
)
display(HTML(anim_attn.to_jshtml(fps=2)))

### Zooming Into the Loss: Per-Token Detail

`step_loss` above is already the _average_ loss HuggingFace computed internally -- useful for training,
but it hides the position-by-position detail Step 4 actually describes. This cell recomputes that same
cross-entropy manually, one position at a time, using the exact `[:-1]` / `[1:]` alignment from
earlier, so we can see individual token losses instead of one averaged number.


> **PyTorch → Keras:** `F.cross_entropy(shift_logits, shift_labels, reduction="none", ignore_index=-100)` — computes per-position cross-entropy loss manually (instead of the averaged loss HuggingFace returns via `labels=`), with `reduction="none"` keeping one loss value per token and `ignore_index=-100` skipping masked positions entirely. **Keras/TF equivalent:** `tf.keras.losses.SparseCategoricalCrossentropy(reduction='none')` — TF/Keras has no built-in `ignore_index`; masking is instead done by multiplying the per-token loss by a `0`/`1` mask tensor (or passing a matching `sample_weight`) built from the same padding/prompt logic.

In [ ]:
# Real per-position loss for the first few real (non-padding) tokens
shift_logits = outputs.logits[0, :-1, :]  # drop the last position's logits (nothing left to predict)
shift_labels = labels[0, 1:]  # drop the first label so index i lines up with logits[i]'s target
per_token_loss = F.cross_entropy(
    shift_logits, shift_labels, reduction="none", ignore_index=-100
)  # unreduced, per-position loss so individual positions can be inspected
positions_to_show = min(8, real_len - 1)
losses_per_pos = per_token_loss[:positions_to_show].detach().float().cpu().numpy()  # move to numpy for printing

print(f"Per-token loss for the first {positions_to_show} real positions:")
print(losses_per_pos.round(3))
print(
    f"Mean of these {positions_to_show}: {losses_per_pos.mean():.3f}  "
    f"(compare to the full-sequence average loss printed above: {step_loss.item():.3f})"
)


### Which Weights Actually Move?

`Qwen/Qwen2.5-0.5B-Instruct` is made of a few distinct kinds of weights (embeddings, attention, MLP, layernorms,
output head), and the three techniques in this notebook don't touch the same ones:

| Component                       | Full Fine-Tuning             | Partial Freezing                          | LoRA                                |
| ------------------------------- | ---------------------------- | ----------------------------------------- | ----------------------------------- |
| Token + position embeddings     | Updated                      | Frozen                                    | Frozen                              |
| Attention (Q/K/V + output proj) | Updated                      | Frozen (early blocks), updated (last few) | Frozen base + small adapter updated |
| MLP / FFN                       | Updated                      | Frozen (early blocks), updated (last few) | Frozen                              |
| LayerNorms                      | Updated                      | Frozen (early blocks), updated (last few) | Frozen                              |
| Output head                     | Updated (tied to embeddings) | Updated                                   | Frozen                              |

`step_loss.backward()` above already populated a real `.grad` on every weight, since this example runs
full fine-tuning (nothing frozen yet) -- exactly the gradient the backprop animation below traces
block by block.


### Watching Backprop Flow, Block by Block

Step 5 says "compute gradients for every parameter," but that's not something that happens all at
once, or in forward order. Backprop walks the computation graph in **reverse**: the chain rule means
block 23's gradient (the one right next to the loss) can be computed immediately, but block 22's
gradient needs block 23's result first, block 21's needs block 22's, and so on -- all the way back to
block 0, next to the embeddings. That's the entire reason it's called "**back**"-propagation.

The animation below computes the real per-block gradient norms directly from `base_model`'s populated
`.grad` tensors, then reveals them one block at a time, in that same reverse order -- block 23 lights
up first, block 0 lights up last.


> **PyTorch → Keras:** `model.named_parameters()` / `p.grad.norm()` / `torch.norm(torch.stack([...]))` — `named_parameters()` iterates every learnable tensor with its dotted name (used here to filter by transformer block index), `.grad` holds the gradient populated by the earlier `.backward()` call, and `.norm()` computes its L2 magnitude; `torch.stack` + `torch.norm` combine several per-parameter norms into one per-block value. **Keras/TF equivalent:** `model.trainable_variables` — TF/Keras exposes trainable weights as a flat list (with `.name` as the equivalent of parameter names) rather than storing gradients on the tensor itself; gradients instead come back as a separate list from `tape.gradient(...)`, and `tf.norm(tf.stack([...]))` is the direct equivalent of the norm/stack combination here.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Recompute one backward pass so this cell remains valid even after a later cell clears .grad.
# No optimizer step is taken, so none of the model's weights change.
base_model.train()
base_model.zero_grad(set_to_none=True)
backprop_loss = base_model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
).loss
backprop_loss.backward()

# Real gradient magnitude per transformer block (actual gradient flow, not a fabricated curve).
model_blocks = base_model.model.layers
n_blocks = len(base_model.model.layers)
assert len(model_blocks) == n_blocks
block_grad_norms = []
for i, block in enumerate(model_blocks):
    block_params = [
        parameter for parameter in block.parameters() if parameter.grad is not None
    ]
    norm = (
        torch.linalg.vector_norm(
            torch.stack([parameter.grad.norm() for parameter in block_params])
        ).item()
        if block_params
        else 0.0
    )
    block_grad_norms.append(norm)

max_grad_norm = max(block_grad_norms)
if not np.isfinite(max_grad_norm) or max_grad_norm <= 0:
    raise RuntimeError("Backprop produced no finite, non-zero block gradients to plot.")

# One frame per block, revealed in reverse dependency order.
block_order = list(range(n_blocks - 1, -1, -1))
fig_bp, ax_bp = plt.subplots(figsize=(9, 6))


def _update_backprop(frame):
    ax_bp.clear()
    current_block = block_order[frame]
    revealed = set(block_order[: frame + 1])
    heights = [block_grad_norms[i] if i in revealed else 0.0 for i in range(n_blocks)]
    colors = [
        "#d62728" if i == current_block else "#9467bd" if i in revealed else "whitesmoke"
        for i in range(n_blocks)
    ]
    ax_bp.barh(np.arange(n_blocks), heights, color=colors)
    ax_bp.set_xlim(0, max_grad_norm * 1.15)
    ax_bp.set_ylim(-0.5, n_blocks - 0.5)
    ax_bp.invert_yaxis()
    ax_bp.set_xlabel("Real gradient norm", fontsize=9)
    ax_bp.set_ylabel(
        f"Block (0 = nearest embeddings, {n_blocks - 1} = nearest the loss)", fontsize=8
    )
    ax_bp.set_title(
        f"Backprop step {frame + 1}/{n_blocks}: gradient just reached block {current_block}",
        fontsize=10,
        fontweight="bold",
    )


anim_backprop = FuncAnimation(
    fig_bp, _update_backprop, frames=n_blocks, interval=180, blit=False
)
plt.close(fig_bp)  # prevent a duplicate static frame from also rendering

print(
    f"Animation uses real block gradient norms from {min(block_grad_norms):.3e} "
    f"to {max_grad_norm:.3e}. Red = current block; purple = already revealed; "
    "gray = not reached yet."
)
display(HTML(anim_backprop.to_jshtml(fps=5)))

# Leave the shared model clean for any cells run afterward.
base_model.zero_grad(set_to_none=True)
_ = base_model.eval()

### Step 6: The Actual Weight Update

This is the step every fine-tuning technique in this notebook is really about:
`W_new = W_old - learning_rate × gradient`. We're not letting an optimizer do this at scale yet (that
happens inside `Trainer.train()` in the very next section) -- instead, we manually apply that same
formula to one real weight from `base_model`, so the update is visible instead of buried inside
thousands of simultaneous parameter updates. One nudge is tiny: `lr=5e-5` times a small gradient often
works out to around `1e-8`, far too small to notice at 6 decimal places, which is why the printed delta
below uses scientific notation. `base_model.zero_grad()` and `.eval()` afterward reset the model back
to exactly how the rest of the notebook expects to find it -- this was a one-off illustration, not a
real training step, so nothing here is meant to persist. Once that's done, let's put all six steps
into one figure below.


> **PyTorch → Keras:** `sample_param.data` / `sample_param.grad` — `.data` accesses a parameter's raw tensor values while bypassing autograd tracking (used here purely to read/print a value), and `.grad` reads the gradient populated by the earlier backward pass; the manual `new_weight = old_weight + weight_delta` line reproduces the SGD update rule `W_new = W_old - lr × gradient` by hand instead of calling an optimizer. **Keras/TF equivalent:** `variable.numpy()` / `tape.gradient(...)` — a Keras/TF version would read `variable.numpy()` for the raw value and the corresponding entry from `tape.gradient(loss, model.trainable_variables)` for the gradient, then apply the same formula manually, or just call `optimizer.apply_gradients(...)` for the real (non-illustrative) update.

In [ ]:
# Recompute a backward pass so this demonstration does not depend on gradients surviving
# from an earlier cell. No optimizer step is taken, so the model weights remain unchanged.
base_model.train()
base_model.zero_grad(set_to_none=True)
weight_demo_loss = base_model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
).loss
weight_demo_loss.backward()

# Keep only matrix parameters with a finite, nonzero gradient, then choose the matrix
# containing the strongest scalar gradient so the tiny update is still visible.
gradient_candidates = [
    (name, parameter)
    for name, parameter in base_model.named_parameters()
    if parameter.dim() == 2
    and parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    and torch.count_nonzero(parameter.grad).item() > 0
]
if not gradient_candidates:
    raise RuntimeError("Backward pass produced no finite, nonzero matrix gradients.")

sample_name, sample_param = max(
    gradient_candidates,
    key=lambda item: item[1].grad.abs().max().item(),
)
sample_index = sample_param.grad.abs().argmax().item()
old_weight = sample_param.data.flatten()[sample_index].item()
sample_grad = sample_param.grad.flatten()[sample_index].item()

# Calculate one real SGD update without assigning it back to the parameter.
lr = 5e-5
weight_delta = -lr * sample_grad
new_weight = old_weight + weight_delta

# Remove demonstration gradients and restore inference mode for downstream cells.
base_model.zero_grad(set_to_none=True)
base_model.eval()

print(f"Parameter: {sample_name} (flat index {sample_index})")
print(f"W_old = {old_weight:.6f}   gradient = {sample_grad:.3e}   LR = {lr:.0e}")
print(f"ΔW = -lr * gradient = {weight_delta:+.3e}   ->  W_new = {new_weight:.6f}")

### Watching the Optimizer Nudge, Zoomed In

Step 6's formula, `W_new = W_old - lr × gradient`, produced a real number above -- but at 6 decimal
places the change was invisible, which is exactly why the cell above had to print it in scientific
notation. The animation below plots those same `W_old`/`W_new` values, zoomed into a window just
barely wide enough to fit both, so the real (tiny) movement becomes visible instead of implied. Nothing
here is exaggerated or fabricated -- only zoomed in.

> **What `W_old` and `W_new` actually are here**
>
> These are **a single floating-point number** — specifically, the `.flatten()[0]` element (the very
> first weight in the matrix when laid out in row-major order) of the **first 2D parameter that had
> a non-null gradient** (found by `next(...)` in the cell above, whose name is printed as
> `"Parameter: ..."` when that cell runs). That one scalar was chosen purely to make the update
> _visible_; it is not a per-block average, a norm, or any kind of aggregate.
>
> The same `W_new = W_old - lr × gradient` formula is simultaneously applied to every one of
> `Qwen/Qwen2.5-0.5B-Instruct`'s ~the runtime-reported individual weights during a real optimizer step — this animation just zooms
> in on one of them so the mechanics are tangible. The per-block weight-delta chart further down in
> this notebook shows the _spread_ of those simultaneous updates across all 24 transformer blocks.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Animate the real W_old -> W_new nudge from the cell above. The axis is zoomed tightly around the
# two real values (not an arbitrary exaggeration) so the genuinely tiny movement is actually visible.
n_frames_update = 20
pad = abs(weight_delta) * 1.5 if weight_delta != 0 else 1e-9
axis_lo, axis_hi = min(old_weight, new_weight) - pad, max(old_weight, new_weight) + pad

fig_update, ax_update = plt.subplots(figsize=(10, 3))


def _update_weight_anim(frame):
    ax_update.clear()
    t = frame / (n_frames_update - 1)
    current_value = old_weight + t * weight_delta
    ax_update.axvline(
        old_weight, color="steelblue", linestyle="--", linewidth=1.5, label="W_old"
    )
    ax_update.axvline(
        new_weight, color="green", linestyle="--", linewidth=1.5, label="W_new"
    )
    ax_update.plot(
        [current_value], [0], marker="o", markersize=14, color="darkred", zorder=5
    )
    ax_update.set_xlim(axis_lo, axis_hi)
    ax_update.set_ylim(-1, 1)
    ax_update.set_yticks([])
    ax_update.set_xlabel(
        f"Weight value, zoomed to the real W_old/W_new window (parameter: {sample_name})",
        fontsize=8,
    )
    ax_update.legend(loc="upper left", fontsize=8)
    ax_update.set_title(
        f"Step 6: applying \u0394W = -lr \u00d7 gradient -- {t:.0%} of the way there\n"
        f"current = {current_value:.10f}  (W_old={old_weight:.10f}, W_new={new_weight:.10f})",
        fontsize=9,
        fontweight="bold",
    )


anim_update = FuncAnimation(
    fig_update, _update_weight_anim, frames=n_frames_update, interval=120, blit=False
)
plt.close(fig_update)  # prevent a duplicate static frame from also rendering

print(
    f"Animation: the red dot slides from W_old to W_new -- the same real numbers printed above "
    f"(\u0394W = {weight_delta:+.3e}), plotted on an axis zoomed tightly around the two values so the "
    f"movement is actually visible. At normal scale this nudge is far too small to see, which is "
    f"exactly why fine-tuning needs thousands of steps, not one, to add up to real learning.\n"
)
display(HTML(anim_update.to_jshtml(fps=8)))

In [ ]:
# Anatomy of one training step -- all six real numbers above, laid out in one figure
# (not fabricated numbers: every value below came from the cells above, computed for real on base_model)
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Anatomy of One Training Step (Continued Pretraining) -- real numbers from Qwen/Qwen2.5-0.5B-Instruct",
    fontsize=13,
    fontweight="bold",
)

# Step 1: Tokenization
ax1 = axes[0, 0]
ax1.text(
    0.5,
    0.88,
    "Step 1: Tokenization",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax1.transAxes,
)
ax1.text(
    0.5,
    0.45,
    f'"{example_text[:38]}..."\n\u2193\n{input_ids[0, :8].tolist()} ...',
    ha="center",
    va="center",
    fontsize=9,
    transform=ax1.transAxes,
    bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.7),
)
ax1.axis("off")
ax1.legend(
    handles=[
        Patch(
            facecolor="lightblue",
            alpha=0.7,
            edgecolor="black",
            label="Text -> token ids",
        )
    ],
    loc="lower center",
    fontsize=7,
)

# Step 2: The mask layout (real tokens vs. padding) -- the actual answer to "how is the mask laid out"
ax2 = axes[0, 1]
mask_row = attention_mask[0].cpu().numpy().reshape(1, -1)
ax2.imshow(
    mask_row, cmap="Greens", aspect="auto", vmin=0, vmax=1, extent=[0, 128, 0, 1]
)
boundary_line = ax2.axvline(
    real_len, color="red", linestyle="--", linewidth=1.5, label="Real/padding boundary"
)
ax2.set_yticks([])
ax2.set_xlabel("Token position (0-128)", fontsize=8)
ax2.set_title(
    f"Step 2: Mask Layout\n{real_len} real tokens (green, active) +\n"
    f"{128 - real_len} padding (white, labels=-100)",
    fontsize=9,
    fontweight="bold",
)
ax2.legend(
    handles=[
        Patch(facecolor="darkgreen", label="Real token (active)"),
        Patch(facecolor="white", edgecolor="black", label="Padding (labels=-100)"),
        boundary_line,
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=1,
    fontsize=7,
)

# Step 3: Forward pass -- NOT the raw logit values (those are just arbitrary numbers to look at).
# What actually matters for "forward pass" is the causal mask that lets one parallel call stand in
# for 128 sequential ones: position i's row can only see columns j <= i.
ax3 = axes[0, 2]
mask_window = min(16, real_len)  # small enough that individual cells are still readable
causal_mask_matrix = np.tril(
    np.ones((mask_window, mask_window))
)  # 1 = i can attend to j
ax3.imshow(causal_mask_matrix, cmap="Greens", vmin=0, vmax=1, aspect="equal")
ax3.set_xlabel("Position j (key)", fontsize=8)
ax3.set_ylabel("Position i (query)", fontsize=8)
ax3.set_title(
    f"Step 3: Forward Pass\ncausal mask, first {mask_window} positions",
    fontsize=9,
    fontweight="bold",
)
ax3.set_xticks(range(0, mask_window, max(1, mask_window // 4)))
ax3.set_yticks(range(0, mask_window, max(1, mask_window // 4)))
ax3.legend(
    handles=[
        Patch(facecolor="darkgreen", label="i can attend to j (j \u2264 i)"),
        Patch(
            facecolor="white", edgecolor="black", label="blocked (j > i, the future)"
        ),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    fontsize=6.5,
)

# Step 4: Real per-position loss
ax4 = axes[1, 0]
positions = np.arange(len(losses_per_pos))
ax4.bar(
    positions,
    losses_per_pos,
    color="coral",
    alpha=0.8,
    width=0.6,
    label="Per-token loss",
)
ax4.axhline(
    losses_per_pos.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Mean (shown)",
)
ax4.set_xlabel("Token position", fontsize=9)
ax4.set_ylabel("Loss", fontsize=9)
ax4.set_title(
    f"Step 4: Compute Loss\nfull-sequence avg = {step_loss.item():.3f}",
    fontsize=9,
    fontweight="bold",
)
ax4.legend(fontsize=8)

# Step 5: Real gradient magnitude per block
ax5 = axes[1, 1]
tick_stride = max(1, n_blocks // 8)
ax5.barh(
    np.arange(n_blocks),
    block_grad_norms,
    color="purple",
    alpha=0.7,
    label="Gradient norm per block",
)
ax5.set_yticks(np.arange(0, n_blocks, tick_stride))
ax5.set_yticklabels([f"Block {i}" for i in range(0, n_blocks, tick_stride)], fontsize=8)
ax5.set_xlabel("Real gradient norm", fontsize=9)
ax5.set_title(
    "Step 5: Backpropagation\nactual per-block gradient norm",
    fontsize=9,
    fontweight="bold",
)
ax5.invert_yaxis()
ax5.legend(fontsize=7, loc="lower right")

# Step 6: Real weight update -- shown at enough precision to actually see the nudge
ax6 = axes[1, 2]
ax6.text(
    0.5,
    0.88,
    "Step 6: Update Weights",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.58,
    f"W_old = {old_weight:.6f}\ngradient = {sample_grad:.3e}\nLR = {lr:.0e}",
    ha="center",
    va="center",
    fontsize=10,
    transform=ax6.transAxes,
)
ax6.text(
    0.5,
    0.28,
    f"\u0394W = {weight_delta:+.3e}\nW_new = {new_weight:.6f}",
    ha="center",
    va="center",
    fontsize=11,
    transform=ax6.transAxes,
    color="green",
    fontweight="bold",
    bbox=dict(boxstyle="round", facecolor="lightgreen", alpha=0.5),
)
ax6.axis("off")
ax6.legend(
    handles=[
        Patch(
            facecolor="lightgreen",
            alpha=0.5,
            edgecolor="black",
            label="New (updated) weight",
        )
    ],
    loc="lower center",
    fontsize=7,
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"\n{'=' * 80}")
print(
    "Training Step Summary (all numbers above are real, from one forward+backward pass):"
)
print(f"{'=' * 80}")
print(f"1. Tokenize: {real_len} real tokens + {128 - real_len} padding tokens")
print("2. Mask layout: labels = input shifted left; padding positions set to -100")
print(f"3. Forward pass: logits shape {tuple(outputs.logits.shape)}")
print(f"4. Loss (avg over real tokens only): {step_loss.item():.3f}")
print(f"5. Backprop: gradients computed for all {n_blocks} transformer blocks")
print(
    f"6. Update: W_new = W_old + \u0394W, where \u0394W = -lr * gradient = {weight_delta:+.3e}"
)
print(
    "   That's why fine-tuning needs many steps: each one nudges a weight by a fraction of a "
    "percent, and Riverside's assistant only 'learns' after thousands of these tiny nudges add up."
)
print(f"{'=' * 80}")

---

## The Fine-Tuning Journey: Let Each Failure Choose the Next Objective

```mermaid
flowchart TD
    A["Base model<br/>fluent, catalog-blind"] -->|"generic Riverside prose"| B["Continued pretraining"]
    B -->|"continues requests instead of obeying them"| C["SFT"]
    C -->|"valid answer, wrong editorial choice"| D["DPO or another preference method"]
```

This is a diagnostic sequence, not mandatory checkpoint ancestry. Stop as soon as Riverside's required behavior passes its acceptance probes.

Part 1 changes the **training experience**. Part 2 separately asks whether full fine-tuning, freezing, LoRA, or QLoRA can carry that experience within Riverside's hardware and release constraints.

## Concept 1: Continued Pretraining - Make Riverside Prose Less Surprising

The baseline failed on Riverside-only language because those names, relationships, and stylistic patterns were absent from its training experience.

**Minimal fix:** keep the original next-token objective, but continue training on raw Riverside paragraphs. There are no instructions or preference labels yet; the model simply predicts the next manuscript token. This is **continued pretraining**, also called domain-adaptive pretraining.

| What this experience can teach | What it cannot teach |
| --- | --- |
| Domain vocabulary, recurring entities, prose patterns | How to obey a user request |
| Which continuations resemble Riverside text | When to stop or return a required format |
| A better prior for later adaptation | Which of two acceptable answers an editor prefers |

The runnable example updates all model weights so the learning signal is easy to inspect. Part 2 will challenge that expensive parameter choice.

**Checkpoint after training:** rerun the catalog-fluency probe, then issue a bounded request such as `Answer in one sentence and stop.` If the continuation becomes more Riverside-like but the model still treats the request as text to continue, continued pretraining worked and exposed the next blocker.

> **Bridge to SFT:** the model has practiced Riverside prose, not the interaction contract of an assistant. The next section exists because knowing the domain and following an instruction are different behaviors.

### Code Walkthrough: `tokenize_causal()` — Preparing Text for Next-Token Prediction

This is the first point in the notebook where we actually need to convert raw paragraph strings into
the fixed-length integer tensors a transformer consumes, so this is where `tokenize_causal()` gets
defined, right before the `dataset.map(...)` call that needs it. Every later stage that trains on
plain continuation text (partial freezing and LoRA continued pretraining, further down) reuses this
exact same function; the instruction-tuning and DPO sections swap in a response-masked variant
instead, since those need to hide the prompt from the loss.

```python
def tokenize_causal(examples, tokenizer, max_length=128):
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens
```

**Arguments:**

| Argument     | Type                  | Purpose                                                                                                                                                                                                                                        |
| ------------ | --------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `examples`   | `dict` (HF batch)     | A batch of examples with a `"text"` column — here, the raw paragraph strings in `non_inst_paragraphs`, wrapped in a `Dataset`.                                                                                                                 |
| `tokenizer`  | `PreTrainedTokenizer` | The tokenizer loaded in the baseline cell above (`AutoTokenizer.from_pretrained(MODEL_NAME)`). Passed in explicitly rather than closed over, so the same function works unchanged no matter which model/tokenizer this notebook is pointed at. |
| `max_length` | `int`, default `128`  | Hard cap on sequence length. Longer paragraphs are truncated; shorter ones are padded up to this length so every example in a batch has the same shape.                                                                                        |

**What it returns:** the usual tokenizer output (`input_ids`, `attention_mask`) plus a `labels` key,
which HuggingFace's `Trainer` requires to compute the causal-LM loss. `labels` starts as a copy of
`input_ids`, then every padding position (where `attention_mask == 0`) is overwritten with `-100` —
PyTorch's `CrossEntropyLoss` convention for "ignore this position." Without that mask, the model would
waste training signal learning to predict padding tokens instead of real text.

It's called in the cell below as `dataset.map(lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"])`.
`batched=True` is what makes `examples` a dict-of-lists (every text in the batch at once) instead of a
single example, which is why the list comprehension inside zips over `tokens["input_ids"]` rather than
indexing a single sequence.


### Optional Depth: Long Documents, Truncation, and Packing

Plain `truncation=True` is a paper cutter: without overflow handling, an 800-token example capped at 512 contributes only its first 512 tokens.

This notebook's `tokenize_causal()` also sets `return_overflowing_tokens=True`, so a long paragraph becomes multiple fixed-length chunks instead of silently losing its tail. The final short chunk is padded and its padding labels are masked.

| Training type | Common strategy | Trade-off |
| --- | --- | --- |
| SFT | Truncate or separately budget prompt and completion | Preserves pair structure, but an overlong response may still lose its tail |
| Continued pretraining | Overflow chunks or pack documents into fixed blocks | Preserves more text, but block boundaries weaken cross-boundary context |

```text
BLOCK A: [ Token 0 ... Token 127 ]
BLOCK B: [ Token 128 ... ]  <- attention starts again here
```

The first tokens in Block B cannot attend to Block A even when they continue the same paragraph. Production pipelines may use document-aware packing, block-diagonal attention, best-fit grouping, or local overlap to manage that trade-off.

For this teaching run, overflow chunks keep every paragraph tail visible while preserving a simple fixed-length loss mask.

> **PyTorch → Keras:** `from datasets import Dataset` / `from transformers import Trainer, TrainingArguments` / `dataset.map(...)` — HuggingFace's `Dataset.map()` applies `tokenize_causal()` to every example (batched, for speed), producing the `input_ids`/`attention_mask`/`labels` columns that `Trainer` (a full PyTorch training-loop wrapper: batching, forward/backward, optimizer step) consumes next. **Keras/TF equivalent:** `tf.data.Dataset.map(...)` + `model.fit(...)` — a Keras version would build a `tf.data.Dataset` pipeline with the same `.map()` call and then call the standard `model.fit(dataset, epochs=...)` in place of HuggingFace's `Trainer` (or use `TFAutoModelForCausalLM` with HuggingFace's own `Trainer`, which wraps `model.fit` under the hood).

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

non_inst_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery"],
    max_chapters=None,  # None = all available chapters per novel
)
print(
    f"Loaded {len(non_inst_paragraphs)} paragraphs from 3 novels for continued-pretraining demo"
)

non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})  # wrap the paragraph list in a HF Dataset


def tokenize_causal(examples, tokenizer, max_length=128):
    """Standard next-token-prediction tokenization: labels = input_ids, with padding
    positions masked out (-100) so the loss ignores them."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length, return_overflowing_tokens=True
    )
    # return_overflowing_tokens=True ensures that long texts are split into multiple chunks, each of max_length tokens
    # The last chunk with fewer than max_length tokens will still be included, padded to max_length
    # This ensures tokenizer is never dropping text due to length constraints

    # Copy input ids into labels, replacing padding positions with -100 so the loss skips them
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


non_inst_tokenized = non_inst_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)  # apply tokenize_causal across the whole dataset in batches, dropping the raw text column


Data's ready. Now load a **fresh, untouched copy** of `Qwen/Qwen2.5-0.5B-Instruct` to actually fine-tune -- kept
separate from `base_model` so `base_model` stays the permanent "before" snapshot every later
comparison in this notebook relies on.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` / `p.numel()` — loads a second, independent copy of `Qwen/Qwen2.5-0.5B-Instruct` (kept separate from `base_model` so the original stays an untouched "before" snapshot) and `.numel()` counts the total scalar elements in each parameter tensor to report the full trainable-parameter count. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` / `model.count_params()` — Keras models expose a built-in `count_params()` method that sums all trainable + non-trainable weight sizes in one call, instead of manually summing `p.numel()` over every parameter.

In [ ]:
full_ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)  # separate fresh copy dedicated to full fine-tuning
print(
    f"Loaded a fresh {MODEL_NAME} to fully fine-tune: "
    f"{sum(p.numel() for p in full_ft_model.parameters()):,} parameters, all trainable."
)


### Configuring and Running the Trainer

`TrainingArguments` + `Trainer` is HuggingFace's standard training loop -- it handles the batching,
the forward/backward pass, and the optimizer step described earlier in this notebook, so we don't
write that loop by hand. `max_steps=60` and `learning_rate=5e-5` keep this CPU demo fast; a real
Riverside training run would raise `max_steps` substantially. `trainer_full.train()` is the line that
actually runs all 20 of those steps -- this is the real training run every later comparison in this
notebook is measured against.


> **PyTorch → Keras:** `TrainingArguments(...)` / `Trainer(model=..., args=..., train_dataset=...)` / `trainer_full.train()` / `full_ft_model.save_pretrained(...)` — configures and runs HuggingFace's full PyTorch training loop (batching, forward/backward passes, optimizer steps, logging) in one `.train()` call, then serializes the fine-tuned weights + config to disk. **Keras/TF equivalent:** `model.compile(optimizer=..., loss=...)` + `model.fit(dataset, epochs=...)` — the direct Keras analog of configuring + running training; `save_pretrained(...)` has an identically-named method on `TFPreTrainedModel` subclasses, so the checkpoint-saving line itself would be unchanged in a TF version.

In [ ]:
# Configure a short training run (max_steps kept small for CPU-friendly demo purposes)
training_args_full = TrainingArguments(
    output_dir="./checkpoints/non-instruction-full",
    per_device_train_batch_size=2,
    # number of steps during gradient descent, kept small for CPU-friendly demo purposes, in practice this would be around several thousand
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=10,
    save_strategy="no",
    learning_rate=5e-5,
    report_to="none",
)

# Wrap the model, config, and tokenized dataset in a Trainer and run the actual training loop
trainer_full = Trainer(
    model=full_ft_model, args=training_args_full, train_dataset=non_inst_tokenized
)
trainer_full.train()
full_ft_model.save_pretrained("./checkpoints/non-instruction-full")  # persist weights to disk for later reload
print("Saved continued-pretraining (full fine-tune) checkpoint.")


### What the Fully Fine-Tuned Model Actually Generates

Training loss shows that the model updated, but it does not show what changed at inference time. The next cell runs the same three held-out health-check prompts through both the untouched base model and the newly trained checkpoint. This makes domain adaptation, general-language retention, and possible memorization visible rather than inferred from the loss curve alone.

In [ ]:
# Compare real continuations before and after full continued pretraining.
full_training_prompts = {
    "Domain knowledge": "Aria Voss checked the Meridian's Promise and",
    "General knowledge retention": "The capital of France is",
    "Novel domain generalization": "In the Under-Hold, the rebels gathered and",
}

for label, prompt in full_training_prompts.items():
    print(f"=== {label} ===")
    print(f"Prompt : {prompt!r}")

    # manual seed controls which random numbers
    # are used for the initial values of the model's parameters.
    # a random seed generates a fixed set of random numbers that can be reproduced across runs
    # this ensures that the generated outputs are consistent and comparable across runs
    torch.manual_seed(42)
    print(f"Base   : {generate(base_model, prompt)}")
    torch.manual_seed(42)
    print(f"Trained: {generate(full_ft_model, prompt)}")
    print()

### Weight Movement Layer by Layer

The bar chart above shows each block's _total_ movement collapsed into one number - the L2 norm of all
its deltas combined. That is useful for comparing blocks, but it loses the distribution of movement
_within_ each block and does not show whether large and small changes are clustered or uniformly
spread.

The cell below samples the same number of individual weight deltas from every transformer block, then
gives **each block its own panel**. Figures contain at most 10 panels, so the runtime-reported decoder blocks are paginated automatically. Every panel uses the same y-axis scale: a quiet block therefore
cannot look as active as a strongly moving block merely because its axis was automatically rescaled.

> **What to look for:** Compare the mean and maximum in each panel title, then inspect the shape. Long
> regions near zero mean many sampled weights barely moved; spikes identify sampled weights with larger
> updates. A concentration of higher means or taller spikes in later blocks suggests those layers
> adapted more strongly during continued pretraining.

In [ ]:
# Load the saved checkpoint (full_ft_model was freed above; reload from disk for the layer panels)
# Snapshot the pre-fine-tune weights (base_model is the permanent "before" snapshot), so
# we can diff against them below.
import gc

base_state = dict(base_model.named_parameters())
ft_trace_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to("cpu")

SAMPLES_PER_BLOCK = 500  # weights sampled per transformer block
PANELS_PER_FIGURE = 10
N_COLUMNS = 2

all_deltas = []
for block_i in range(len(base_model.model.layers)):  # walk every transformer block in order
    block_deltas = []
    for name, parameter in ft_trace_model.named_parameters():
        if f"model.layers.{block_i}." in name:  # only this block's own parameters
            delta = (
                parameter.data.cpu() - base_state[name].data.cpu()
            ).abs().flatten()  # absolute weight movement vs. the untouched base checkpoint
            block_deltas.append(delta)
    if block_deltas:
        combined = torch.cat(block_deltas)
        stride = max(1, len(combined) // SAMPLES_PER_BLOCK)  # even subsampling so every block plots the same count
        sampled = combined[::stride][:SAMPLES_PER_BLOCK].numpy()
        all_deltas.append(sampled)

# Use one shared scale across every figure. Without this, a quiet block could look as active as
# the block with the largest movement simply because Matplotlib rescaled its panel.
global_ymax = max(float(block.max()) for block in all_deltas)
y_limit = global_ymax * 1.05 if global_ymax > 0 else 1e-9

for page_start in range(0, len(all_deltas), PANELS_PER_FIGURE):  # paginate blocks across multiple figures
    page = all_deltas[page_start : page_start + PANELS_PER_FIGURE]
    n_rows = (len(page) + N_COLUMNS - 1) // N_COLUMNS

    fig, axes = plt.subplots(
        n_rows,
        N_COLUMNS,
        figsize=(14, 2.6 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()

    for panel_i, block_arr in enumerate(page):  # one subplot per transformer block on this page
        block_i = page_start + panel_i
        weight_indices = np.arange(len(block_arr))
        axis = axes[panel_i]

        axis.plot(weight_indices, block_arr, linewidth=0.7, color="steelblue")
        axis.fill_between(weight_indices, block_arr, alpha=0.12, color="steelblue")
        axis.set_title(
            f"Transformer block {block_i}  "
            f"(mean={block_arr.mean():.2e}, max={block_arr.max():.2e})",
            fontsize=9,
        )
        axis.set_ylim(0, y_limit)
        axis.grid(alpha=0.2, axis="y")

    for unused_axis in axes[len(page) :]:
        unused_axis.set_visible(False)  # hide any empty grid cells on the last page

    page_end = page_start + len(page) - 1
    fig.suptitle(
        f"Full Fine-Tuning Weight Movement: Blocks {page_start}-{page_end}",
        fontsize=12,
        fontweight="bold",
    )
    fig.supxlabel(f"Sampled weight index ({SAMPLES_PER_BLOCK} weights per block)")
    fig.supylabel("|W_after - W_before|")
    plt.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.show()

peak_block = max(range(len(all_deltas)), key=lambda i: all_deltas[i].mean())  # block with the largest average movement
min_block = min(range(len(all_deltas)), key=lambda i: all_deltas[i].mean())  # block with the smallest average movement
print(
    f"Mean |delta W| by block - min: block {min_block} "
    f"({all_deltas[min_block].mean():.4e}),  "
    f"max: block {peak_block} ({all_deltas[peak_block].mean():.4e}).  "
    f"Each panel shows {SAMPLES_PER_BLOCK} sampled individual-weight deltas."
)

del ft_trace_model
gc.collect()  # free the reloaded model's memory now that its deltas are captured
print("Freed ft_trace_model from memory (checkpoint still on disk).")


### Visualizing Training Progress: Loss Curves

After training completes, let's look at what actually happened -- the real per-step loss recorded by
the `Trainer` above, not an idealized illustration. Textbook loss curves are smooth; a 60-step,
batch-size-2 CPU demo on a fresh model is usually much noisier, and that's worth seeing honestly.

**What to look for:**

1. **Downward trend:** Loss should decrease on average (model is learning), even if noisy step-to-step
2. **Convergence:** Loss should stop trending strongly downward by the end (not still falling fast)
3. **Magnitude:** Lower loss = better fit to domain text (but watch for overfitting on tiny corpora!)

**Reading a noisy real curve:** with only 2 logged points and batch_size=2, a single unusually easy or
hard paragraph can swing the reported loss by ±0.3 or more. Don't over-interpret small wiggles -- look
at the overall direction across all points, and compare against the other techniques' real curves
later in the notebook (instruction tuning, partial freezing, LoRA continued pretraining) to see which
setup is converging fastest for the same step budget.


> **PyTorch → Keras:** `trainer.state.log_history` — HuggingFace's `Trainer` records a running list of dicts (step number, loss, learning rate, etc.) logged every `logging_steps`; this cell filters that list down to just the `(step, loss)` pairs for plotting. **Keras/TF equivalent:** `history = model.fit(...)` / `history.history["loss"]` — Keras's `fit()` returns a `History` object whose `.history` dict holds per-*epoch* (not per-step, by default) metric lists; matching HuggingFace's per-step granularity in Keras needs a custom callback (e.g. overriding `on_train_batch_end`).

In [ ]:
# Visualize the REAL loss curve from the continued-pretraining run above (trainer_full),
# not a fabricated "typical" curve -- this is exactly what your training just did.
def extract_loss_history(trainer):

    # Pull (step, loss) pairs out of the Trainer's log history, skipping eval-only entries
    return [
        (entry["step"], entry["loss"])
        for entry in trainer.state.log_history
        if "loss" in entry
    ]


full_ft_history = extract_loss_history(trainer_full)
steps, losses = zip(*full_ft_history)  # split into two parallel sequences for plotting

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    steps,
    losses,
    marker="o",
    linewidth=2,
    markersize=7,
    color="green",
    label="Training loss",
)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title(
    f"Continued Pretraining (Full FT): real loss log ({losses[0]:.2f} \u2192 {losses[-1]:.2f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n{'=' * 70}")
print("Reading this REAL loss curve (not an idealized one):")
print(f"{'=' * 70}")
print(f"  Logged steps: {list(steps)}")
print(f"  Logged losses: {[round(l, 3) for l in losses]}")
print(f"  First -> last: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"{'=' * 70}")
print("What to look for:")
print(
    "  • A clean, monotonic plateau like a textbook figure is the exception, not the rule --"
)
print(
    "    especially at batch_size=2 with only a handful of steps, loss is dominated by"
)
print("    per-batch noise (which paragraph happened to be in this batch) more than by")
print("    the underlying trend.")
print(
    "  • If the trend is flat/noisy rather than decreasing: raise max_steps, increase the"
)
print(
    "    batch size, or train on more paragraphs so the trend has room to dominate the noise."
)
print(
    "  • Compare this to the loss curves for instruction tuning, partial freezing, and LoRA"
)
print(
    "    continued pretraining further down -- they were all recorded the same real way."
)
print(f"{'=' * 70}")


### Diagnose Continued-Pretraining Failures

| Symptom | Likely cause | First response |
| --- | --- | --- |
| General prompts become nonsense | Too many updates on a narrow corpus | Reduce steps and evaluate domain and general prompts together |
| Training perplexity approaches 1 while held-out perplexity stays high | Memorization | Add diverse text and deduplicate repeated passages |
| Most tokens are padding | `max_length` is much larger than typical paragraphs | Match block length to the observed token-length distribution |
| Training crashes around padding | Causal tokenizer has no pad token | Set `tokenizer.pad_token = tokenizer.eos_token` and mask padding labels with `-100` |

A quick health check needs three probes:

```python
generate(model, "Aria Voss checked the Meridian's Promise and")  # domain fit
generate(model, "The capital of France is")                      # retention
generate(model, "In the Under-Hold, the rebels gathered and")    # generalization
```

A failed general prompt suggests forgetting. A word-for-word held-out continuation suggests memorization. Neither is visible from training loss alone.

## Concept 2: Supervised Fine-Tuning - Teach the Request/Response Contract

Continued pretraining fixed one failure and revealed another. The model can produce Riverside-like prose, but a request such as `Answer in one sentence and stop` is still just more text to continue.

**Observed blocker:** domain fluency is not instruction compliance.

Riverside now supplies demonstrations with two roles:

- **Request:** `Continue this passage with one paragraph in the same style.`
- **Desired response:** the next manuscript paragraph.

The model must read both parts, but only the response is its work product. That creates the need for **prompt masking**: request tokens remain visible as context while their labels become `-100`, so the loss grades response tokens only.

Training on many request/response demonstrations is **supervised fine-tuning (SFT)**. The intuition comes before the implementation:

1. show the task contract;
2. show a response that satisfies it;
3. update the model toward the response, not toward reproducing the request.

This notebook derives a small private dataset from adjacent Riverside paragraphs. That is enough to demonstrate the pipeline and the practiced continuation instruction, but it is not evidence of broad instruction following. A production suite needs varied real editor requests, held-out cases, source-support checks, and explicit pass criteria.

**Checkpoint after training:** rerun the bounded-request probe. If the SFT adapter follows the requested shape, compare several valid responses to the same prompt. SFT can imitate a demonstrated answer, but it does not directly express why one acceptable answer is better than another.

> **Bridge to preference alignment:** once outputs are valid, Riverside's remaining complaint is comparative: concise and useful versus technically correct but bloated. That is not another format rule; it is a preference signal.

This teaching run uses LoRA to fit local hardware. Part 2 separates that parameter choice from the SFT objective.

> **PyTorch → Keras:** `from peft import LoraConfig, get_peft_model, TaskType` — imports HuggingFace's PEFT library, which wraps a PyTorch model's targeted `nn.Linear` layers with low-rank adapter matrices and freezes everything else; the actual wrapping happens a few cells down. **Keras/TF equivalent:** there is no first-party `peft` support for `TFPreTrainedModel`s — the common Keras/TF pattern for parameter-efficient tuning is manual layer freezing (`layer.trainable = False` on all but the last few layers) rather than LoRA adapters, since PEFT's LoRA implementation is PyTorch-only.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# This section defines the instruction task and a function to build instruction-response pairs from novels.
# The same instruction guideline is applied to all novels to maintain consistency in the instruction-response pairs.
# In practice
INSTRUCTION_TASK = "Continue the fiction narrative in the same style."


def build_instruction_pairs(novels=None, max_chapters=None):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "cyberpunk", "literary"]

    pairs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        files = sorted(novel_path.glob("chapter-*.txt"))
        if max_chapters is not None:
            files = files[:max_chapters]
        for path in files:
            paragraphs = [
                paragraph.strip().replace("\n", " ")
                for paragraph in path.read_text(encoding="utf-8").split("\n\n")
                if len(paragraph.strip()) > 200
            ]
            for context, response in zip(paragraphs, paragraphs[1:]):
                instruction = f"{INSTRUCTION_TASK}\n\nContext:\n{context}"
                pairs.append({"instruction": instruction, "response": response})
    return pairs


### Where These Instruction Pairs Actually Come From

This notebook does **not** download or run Mistral, Phi-2, GPT-4o, or any separate pair-generation model. The runnable path constructs examples deterministically from Riverside's existing manuscripts:

```text
paragraph i     -> instruction context
paragraph i + 1 -> desired continuation
```

For each adjacent pair, `build_instruction_pairs()` creates one fixed request:

```text
Continue the fiction narrative in the same style.

Context:
<paragraph i>
```

The response is the next real paragraph. That is why the notebook can build 4,080 examples locally without another model or API call.

| Role | What this notebook uses | Downloaded or executed here? |
| --- | --- | --- |
| Pair construction | Python adjacency logic over Riverside paragraphs | Yes; no model required |
| Model being fine-tuned | `Qwen/Qwen2.5-0.5B-Instruct` from Hugging Face | Yes |
| Optional synthetic-pair generator | An instruction model such as `mistralai/Mistral-7B-Instruct-v0.3` | No |

This extraction approach is useful for teaching prompt masking, LoRA SFT, and checkpoint flow, but it has a narrow teaching signal: every example asks for continuation, and the target is copied from the following paragraph. It does **not** create diverse editing, summarization, question-answering, or style-control instructions.

A production data pipeline could load a separate Hugging Face-hosted instruction model, ask it for structured instruction-response JSON, then validate, deduplicate, provenance-tag, and review those generated examples. That would be a separate data-generation stage with its own runtime and quality evaluation; it is not implied by the code in this notebook.

### Tokenizing With the Prompt-Mask Pattern

`build_instruction_pairs()` gave us `(prompt, completion)` strings; `tokenize_instruction()` converts
one pair into the actual tensors `Trainer` needs, using the same prompt-masking idea introduced
earlier in this notebook: `labels` starts as a full copy of the tokenized text, then every **prompt**
position (not just padding) gets set to `-100`, so the loss only ever grades the completion.


> **PyTorch → Keras:** `tokenize_instruction()` — builds `labels` as a copy of the tokenized `input_ids`, then overwrites *both* the prompt-token positions and the padding positions with `-100`, so a cross-entropy loss with `ignore_index=-100` (used earlier in the notebook) only ever grades the completion tokens. **Keras/TF equivalent:** the same masking logic — a Keras/TF version would build an analogous `labels` array with `-100` (or `0` plus a matching `sample_weight` mask, since TF's `SparseCategoricalCrossentropy` has no built-in `ignore_index`) at prompt+padding positions; the tokenization itself is identical since `AutoTokenizer` is framework-agnostic.

In [ ]:
def tokenize_instruction(example, max_length=160, prompt_max_length=96):
    prompt_text = render_instruction(example["instruction"])
    full_text = render_instruction(example["instruction"], example["response"])
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=prompt_max_length,
    )["input_ids"]
    tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    labels = tokens["input_ids"].copy()
    for index in range(min(len(prompt_ids), len(labels))):
        labels[index] = -100
    for index, mask in enumerate(tokens["attention_mask"]):
        if mask == 0:
            labels[index] = -100
    tokens["labels"] = labels
    return tokens


### Building the Instruction Dataset

Now actually build the pairs from 5 genres and tokenize every one of them with the function above.


In [ ]:
instruction_pairs = build_instruction_pairs(
    novels=["scifi", "fantasy", "mystery", "cyberpunk", "literary"],
    max_chapters=None,  # None = all available chapters
)
print(f"Built {len(instruction_pairs)} instruction pairs from 5 novels")

instruction_dataset = Dataset.from_list(instruction_pairs)  # wrap the prompt/completion pairs in a HF Dataset
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction, remove_columns=["instruction", "response"]
)  # apply the prompt-masking tokenizer across every pair


### A Quick LoRA Preview for This SFT Run

SFT defines **what behavior is taught**. LoRA only changes **where the update is stored**: `get_peft_model()` freezes the base and adds small trainable correction matrices to selected attention projections.

That is enough detail for this chapter. The next code cell uses LoRA so the SFT run fits local hardware; Part 2 derives the low-rank path, measures its parameter budget, and inspects the real matrices.

> **PyTorch → Keras:** `LoraConfig(...)` and `get_peft_model(...)` target the four Qwen2.5 attention
projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and report the resulting trainable fraction.
PEFT's wrapping is PyTorch-specific; a Keras implementation needs a compatible low-rank layer wrapper or
manual custom layers rather than coarse whole-layer freezing.


In [ ]:
# LoRA hyperparameters: rank-8 adapters on all four attention projections.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()


### Training and Saving the Adapter

Same `Trainer` pattern as continued pretraining, just with the LoRA-wrapped model, the prompt-masked
dataset, and a higher learning rate (`2e-4` vs. `5e-5`) -- LoRA needs a higher LR since it's only
updating a tiny slice of parameters.


> **PyTorch → Keras:** `Trainer(model=instruct_lora_model, ...)` / `trainer_instruct.train()` / `instruct_lora_model.save_pretrained(...)` — the same HuggingFace `Trainer` pattern as the earlier full fine-tuning run, just pointed at the LoRA-wrapped model and the prompt-masked instruction dataset, with a higher learning rate since only the small adapter matrices are being updated. **Keras/TF equivalent:** `model.fit(dataset, epochs=...)` — as with the earlier full fine-tuning cell, a Keras/TF version would call `.fit()` on the (layer-frozen) model instead of `Trainer.train()`; `save_pretrained()` again has an identically-named counterpart on `TFPreTrainedModel`.

In [ ]:
# Configure a short LoRA training run with a higher LR (only the adapter matrices are trainable)
training_args_instruct = TrainingArguments(
    output_dir="./checkpoints/instruction-lora",
    per_device_train_batch_size=2,
    max_steps=DEMO_TRAIN_STEPS,  # short demonstration run; tune this from measured convergence
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

# Wrap the LoRA-wrapped model, config, and tokenized instruction dataset in a Trainer and run it
trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained("./checkpoints/instruction-lora")  # persist the adapter weights only
print("Saved instruction-tuned LoRA adapter.")


### Instruction Tuning, Recapped

The preceding cells implement one SFT pipeline:

1. `build_instruction_pairs()` turns adjacent paragraphs into an editor request and desired continuation.
2. `tokenize_instruction()` keeps the request visible but masks its labels with `-100`, so loss grades only the assistant response.
3. A LoRA wrapper keeps the base frozen and stores this teaching run's update in a small adapter.
4. `Trainer.train()` batches the examples and updates only the adapter parameters.

```text
Continued pretraining: [labels for text .....................] [pad: -100]
Instruction tuning:   [prompt: -100 ........] [completion labels] [pad: -100]
```

The conceptual change is the supervision boundary: Riverside now grades what the assistant should return for a request instead of every input token.

### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only
**padding** was masked, and every real token was active. Here, the **prompt itself is masked too** --
the model is only ever penalized for generating the completion, never for reproducing the prompt it
was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it
through the real `tokenize_instruction()` used for training, and colors every token position by what
the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)

# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

# Visualize the mask layout as a single color-coded strip (gray=prompt, green=completion, white=padding)
fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)

# Legend entries matching each color band in the strip above
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {render_instruction(example_pair['instruction'])[:80]!r}...")
print(f"Completion text: {example_pair['response'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

### Diagnose Instruction-Tuning Failures

| Symptom | Likely cause | First response |
| --- | --- | --- |
| Model echoes or predicts the request | Prompt tokens were included in the loss | Set prompt and padding labels to `-100` |
| Works only with one exact prefix | Template overfitting | Diversify training instructions or enforce the same production template |
| Responses stop much too early or run too long | Completion-length distribution mismatches the task | Build examples with production-like response lengths |
| LoRA barely learns | Learning rate is too low for the small trainable state | Tune the adapter learning rate; do not inherit full-FT defaults blindly |

Typical starting ranges are `5e-5` to `1e-4` for full fine-tuning, `1e-4` to `2e-4` for partial freezing, and `2e-4` to `5e-4` for LoRA. They are search ranges, not guarantees.

The next code cell compares the trained model with and without the expected prefix, then tests a novel request. That distinguishes template dependence from broader failure to generalize.

In [ ]:
# Quick health check after instruction tuning

instruction_prompt = render_instruction(
    "Continue the fiction narrative in the same style.\n\nContext:\nAria checked the Meridian and"
)
print("=== Test 1: Trained instruction ===")
print(f"  Input : {instruction_prompt!r}")
print(f"  Output: {generate(instruct_lora_model, instruction_prompt)}")
print()

raw_prompt = "Aria checked the Meridian and"
print("=== Test 2: Plain text (checks template dependence) ===")
print(f"  Input : {raw_prompt!r}")
print(f"  Output: {generate(instruct_lora_model, raw_prompt)}")
print()

novel_prompt = render_instruction(
    "Continue the fiction narrative in the same style.\n\nContext:\nIn the Upper decks, Marcus"
)
print("=== Test 3: Novel instruction (generalization check) ===")
print(f"  Input : {novel_prompt!r}")
print(f"  Output: {generate(instruct_lora_model, novel_prompt)}")


## Concept 3: Preference Alignment - When Imitation Is Not Enough

SFT fixed the interaction contract. It still leaves Riverside with a subtler failure: two responses can both obey the request while one is clearly more useful to an editor.

For the same prompt:

- **Chosen $y^+$:** a concise continuation that advances the scene and preserves supplied facts.
- **Rejected $y^-$:** a grammatical continuation that repeats details or buries the useful information.

SFT says, “make this demonstrated answer likely.” It does not directly say, “make this answer more likely **than that competing answer to the same prompt**.” Preference optimization starts with that missing comparison.

### Why not only suppress the rejected response?

If training only pushes $y^-$ down, the objective specifies what to avoid but not what should replace it. The model can move probability toward arbitrary alternatives, become unnaturally terse, exploit response length, or drift away from useful SFT behavior.

The problem is underdetermined: infinitely many policy changes make one rejected sequence less likely, and most are not the editor's intended improvement. We need both a direction and an anchor.

### The preference-data contract

| Field | Meaning |
| --- | --- |
| `prompt` | One shared editor request $x$ |
| `chosen` | Response $y^+$ preferred by the annotator |
| `rejected` | Less-preferred response $y^-$ to the same prompt |

Labels can come from blinded editor comparisons, accept-versus-rewrite behavior, or a calibrated judge. This notebook uses a structural proxy: the real next paragraph is chosen and an unrelated paragraph is rejected. That proves the mechanics, not real editorial alignment.

### The frozen SFT anchor

The frozen reference is an **anchor for measuring relative movement**: instead of asking only how likely a response is now, DPO asks how its likelihood changed from the accepted SFT starting point. Preference training therefore starts with two identical SFT policies:

1. **Policy $\pi_\theta$:** trainable.
2. **Reference $\pi_{\mathrm{ref}}$:** frozen snapshot of the same SFT model.

For a complete response $y$, the log-ratio

$$
\log\frac{\pi_\theta(y\mid x)}{\pi_{\mathrm{ref}}(y\mid x)}
$$

measures movement from the SFT starting point. Positive means the policy increased that response relative to SFT; negative means it decreased it. The comparison we ultimately need is whether the chosen response gained more relative ground than the rejected response.

**Predict:** If policy and reference begin identical, what should both response movements, their difference, the modeled preference probability, and the loss be before the first update?

**Expected:** both movements and their difference are zero, so the modeled probability is $\sigma(0)=0.5$ and the loss is $-\log 0.5=\log 2\approx0.693$. The toy calculation below verifies this neutral starting point before showing one update.

### How DPO Evolved from Reward-Model RLHF

DPO makes more sense after seeing the machinery it removes. Both PPO-based RLHF and DPO begin with the same evidence:

> A human comparison says which response should win. The policy should make that winner gain ground without discarding useful SFT behavior.

```mermaid
flowchart LR
    H["Human comparisons<br/>chosen vs rejected"] --> RM["Train reward model<br/>predict the winner"]
    RM --> PPO["PPO rollouts<br/>optimize learned reward"]
    PPO --> KL["KL anchor<br/>limit drift from SFT"]
    KL --> ID["Reward is represented by<br/>policy movement from reference"]
    ID --> DPO["DPO fits that movement<br/>directly from offline pairs"]
```

#### Step 1: a comparison reveals a reward difference, not an absolute reward

The label $y^+ \succ y^-$ does not say that either response deserves reward 8.4 or -2.1. It says only that the chosen response should score higher. Reward-model RLHF commonly represents this using Bradley-Terry:

$$
P(y^+ \succ y^-\mid x)
=
\sigma\left(r_\phi(x,y^+)-r_\phi(x,y^-)\right).
$$

The reward model $r_\phi$ learns scalar scores whose **difference** predicts the observed winner. Adding the same prompt-dependent constant to both rewards changes neither the pair probability nor the label.

#### Step 2: PPO turns learned reward into a policy

Maximizing a learned reward without restraint invites reward hacking: the policy can exploit blind spots in $r_\phi$ and abandon useful SFT behavior. Classic RLHF therefore asks for high reward and proximity to a frozen SFT reference:

$$
\max_\pi\;
\mathbb{E}_{y\sim\pi(\cdot\mid x)}[r_\phi(x,y)]
-
\beta D_{\mathrm{KL}}\left(\pi(\cdot\mid x)\,\|\,\pi_{\mathrm{ref}}(\cdot\mid x)\right).
$$

PPO is the practical search loop:

1. generate fresh responses from the current policy;
2. score them with the reward model;
3. estimate advantages, usually with a learned value function;
4. apply clipped policy updates;
5. monitor or penalize KL drift;
6. repeat on newly generated behavior.

PPO is powerful because it can explore responses absent from the original pair dataset. That benefit comes with coupled costs DPO will later remove:

- **Four model roles:** policy, frozen reference, reward model, and usually a value head/model.
- **Online generation:** rollouts inside training often dominate elapsed time and infrastructure.
- **Sensitive optimization:** reward scale, clipping, value loss, advantages, KL target, and learning rates interact.
- **Harder failure analysis:** reward hacking, reward-model distribution shift, rollout variance, and policy updates can all explain a regression.

These are general PPO-RLHF tradeoffs, not Riverside-specific objections. The mathematical reason DPO can remove this machinery comes next.

#### Step 3: the KL-regularized optimum exposes a shortcut

For a fixed reward, the policy that exactly solves the KL-regularized objective has the form

$$
\pi^*(y\mid x)
=
\frac{1}{Z(x)}\pi_{\mathrm{ref}}(y\mid x)
\exp\left(\frac{r(x,y)}{\beta}\right).
$$

Read it from right to left: begin with the SFT probability, boost high-reward responses, then renormalize. Rearranging gives

$$
r(x,y)
=
\beta\log\frac{\pi^*(y\mid x)}{\pi_{\mathrm{ref}}(y\mid x)}
+
\beta\log Z(x).
$$

**This is the PPO-to-DPO hinge:** under the ideal KL-regularized optimum, reward and movement from the reference are two representations of the same response ordering. If a pairwise comparison can eliminate the unknown normalizer, the policy can fit that ordering directly.

#### Step 4: pairwise subtraction cancels the unknown normalizer

Chosen and rejected answer the same prompt, so subtracting their represented rewards cancels $\beta\log Z(x)$:

$$
\Delta_\theta
=
\left[\log\pi_\theta(y^+\mid x)-\log\pi_{\mathrm{ref}}(y^+\mid x)\right]
-
\left[\log\pi_\theta(y^-\mid x)-\log\pi_{\mathrm{ref}}(y^-\mid x)\right].
$$

Substitute that anchored movement margin into Bradley-Terry:

$$
\widehat P_\theta(y^+\succ y^-\mid x)=\sigma(\beta\Delta_\theta),
$$

$$
\mathcal L_{\mathrm{DPO}}
=-\log\widehat P_\theta
=\operatorname{softplus}(-\beta\Delta_\theta).
$$

DPO turns preference learning into binary classification whose logit is the chosen response's movement from SFT minus the rejected response's movement from SFT. Here $\beta>0$ is the KL coefficient inherited from the RLHF derivation and the scale applied to the DPO margin. `softplus` is the smooth binary-classification loss $\log(1+e^z)$, so gradients remain defined for every margin.

- $\Delta_\theta<0$: rejected gained more relative ground; loss exceeds $\log 2$.
- $\Delta_\theta=0$: no relative preference movement; probability is $0.5$ and loss is $\log 2$.
- $\Delta_\theta>0$: chosen gained more relative ground; loss falls below $\log 2$.

Crucially, DPO does not require the chosen response's absolute probability to rise. A positive margin can come from raising chosen, lowering rejected, or lowering both while lowering rejected more.

### DPO and PPO-RLHF solve related but different operational problems

| | DPO | PPO-based RLHF |
| --- | --- | --- |
| Policy data | Fixed offline preference pairs | Fresh on-policy rollouts |
| Explicit reward model | No | Yes |
| Value/advantage model | No | Usually |
| Frozen SFT reference | Yes | Yes, for KL control |
| Main strength | Stable, comparatively simple offline optimization | Exploration against rewards on newly generated behavior |
| Main risk | Limited by pair coverage, quality, and biases | Reward hacking, instability, and expensive moving parts |
| Prefer when | Curated pair data captures the target behavior | Online exploration or environment reward is essential |

DPO did not make PPO obsolete. PPO remains useful for verifiable environment rewards, interactive tasks, and objectives that require exploring beyond a fixed pair dataset. DPO is the simpler fit when high-quality offline comparisons already describe the desired behavior.

Before scoring real token sequences, trace this exact margin and loss on a toy distribution over two complete responses.

In [ ]:
# Step 1: expose implicit reward, pair probability, loss, and gradient for two responses.
import torch
import torch.nn.functional as F

DPO_DEMO_BETA = 1.0
reference_logits = torch.log(torch.tensor([0.55, 0.45]))  # [chosen, rejected]


def inspect_toy_dpo(policy_logits, reference_logits, beta):
    """Return every DPO term for a toy distribution over two complete responses."""
    policy_logps = F.log_softmax(policy_logits, dim=0)
    reference_logps = F.log_softmax(reference_logits, dim=0)

    chosen_log_ratio = policy_logps[0] - reference_logps[0]
    rejected_log_ratio = policy_logps[1] - reference_logps[1]
    log_ratios = torch.stack((chosen_log_ratio, rejected_log_ratio))
    implicit_rewards = beta * log_ratios  # identifiable up to one prompt-only constant
    margin = chosen_log_ratio - rejected_log_ratio
    scaled_margin = beta * margin
    modeled_pair_probability = torch.sigmoid(scaled_margin)
    loss = F.softplus(-scaled_margin)
    return {
        "policy_probabilities": policy_logps.exp(),
        "log_ratios": log_ratios,
        "implicit_rewards": implicit_rewards,
        "margin": margin,
        "modeled_pair_probability": modeled_pair_probability,
        "loss": loss,
    }


def print_toy_dpo(label, measurement):
    print(f"\n=== {label} ===")
    print(
        "Current probabilities [chosen, rejected]: "
        f"{measurement['policy_probabilities'].tolist()}"
    )
    print(f"Log-ratio movement from SFT: {measurement['log_ratios'].tolist()}")
    print(f"Implicit rewards (beta * movement): {measurement['implicit_rewards'].tolist()}")
    print(f"Movement margin:             {measurement['margin'].item():+.3f}")
    print(
        "DPO-modeled P(chosen wins): "
        f"{measurement['modeled_pair_probability'].item():.3f}"
    )
    print(f"DPO loss:                    {measurement['loss'].item():.3f}")


# Policy and reference begin as identical SFT snapshots.
initial_policy_logits = reference_logits.clone().requires_grad_(True)
initial = inspect_toy_dpo(initial_policy_logits, reference_logits, DPO_DEMO_BETA)
print_toy_dpo("Identical policy and reference", initial)
assert torch.allclose(initial["implicit_rewards"], torch.zeros(2))
assert torch.allclose(initial["modeled_pair_probability"], torch.tensor(0.5))
assert torch.allclose(initial["loss"], torch.log(torch.tensor(2.0)))
print("Neutral threshold: margin = 0 and loss = log(2); lower loss means chosen gained more relative ground.")

# A positive margin can arise through several valid kinds of relative movement.
print("\n=== Three routes to the same positive margin ===")
movement_examples = {
    "raise chosen only": (0.4, 0.0),
    "lower rejected only": (0.0, -0.4),
    "lower both; reject more": (-0.1, -0.5),
}
for route, (chosen_movement, rejected_movement) in movement_examples.items():
    route_margin = chosen_movement - rejected_movement
    route_probability = torch.sigmoid(torch.tensor(DPO_DEMO_BETA * route_margin))
    route_loss = F.softplus(torch.tensor(-DPO_DEMO_BETA * route_margin))
    print(
        f"{route:26s} movements=({chosen_movement:+.1f}, {rejected_movement:+.1f}) "
        f"margin={route_margin:+.1f} P={route_probability:.3f} loss={route_loss:.3f}"
    )

# Start from a policy that gave the rejected response more relative ground.
policy_logits = torch.log(torch.tensor([0.35, 0.65])).requires_grad_(True)
before = inspect_toy_dpo(policy_logits, reference_logits, DPO_DEMO_BETA)
before["loss"].backward()
print_toy_dpo("Policy moved in the wrong relative direction", before)
print(f"Gradient [chosen, rejected]: {policy_logits.grad.tolist()}")
print("Gradient descent subtracts this gradient: chosen rises and rejected falls in this toy.")

# Take one visible optimization step.
with torch.no_grad():
    policy_logits -= policy_logits.grad
policy_logits.grad = None
after = inspect_toy_dpo(policy_logits, reference_logits, DPO_DEMO_BETA)
print_toy_dpo("After one gradient step", after)

print(
    "\nObserved update:\n"
    f"  margin: {before['margin'].item():+.3f} -> {after['margin'].item():+.3f}\n"
    f"  modeled P: {before['modeled_pair_probability'].item():.3f} -> "
    f"{after['modeled_pair_probability'].item():.3f}\n"
    f"  loss: {before['loss'].item():.3f} -> {after['loss'].item():.3f}"
)
assert after["margin"] > before["margin"]
assert after["loss"] < before["loss"]
print("PASS: DPO increased the anchored chosen-over-rejected advantage.")

In [ ]:
# Step 2: build offline preference triples with one shared prompt and two responses.
# Teaching proxy: adjacent text is always chosen and unrelated text is always rejected.
# Real human preference data is subtler, noisier, and sometimes contradictory.
def build_preference_pairs(novels=None, max_chapters=4, max_pairs=30):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "horror", "literary"]

    chapter_files = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        files = sorted(novel_path.glob("chapter-*.txt"))
        chapter_files.extend(files if max_chapters is None else files[:max_chapters])

    all_paragraphs = []
    for path in chapter_files:
        paragraphs = [
            paragraph.strip().replace("\n", " ")
            for paragraph in path.read_text(encoding="utf-8").split("\n\n")
            if len(paragraph.strip()) > 200
        ]
        if len(paragraphs) >= 2:
            all_paragraphs.append(paragraphs)

    pairs = []
    for chapter_index, paragraphs in enumerate(all_paragraphs):
        other_chapter = all_paragraphs[(chapter_index + 1) % len(all_paragraphs)]
        for paragraph_index in range(len(paragraphs) - 1):
            instruction = (
                f"{INSTRUCTION_TASK}\n\nContext:\n{paragraphs[paragraph_index]}"
            )
            chosen_text = paragraphs[paragraph_index + 1]
            rejected_text = other_chapter[paragraph_index % len(other_chapter)]
            prompt = render_instruction(instruction)
            chosen_full = render_instruction(instruction, chosen_text)
            rejected_full = render_instruction(instruction, rejected_text)
            if not chosen_full.startswith(prompt) or not rejected_full.startswith(prompt):
                raise ValueError("Instruction template did not preserve the DPO prompt prefix")
            pairs.append(
                {
                    "prompt": prompt,
                    "chosen": chosen_full[len(prompt) :],
                    "rejected": rejected_full[len(prompt) :],
                }
            )

    return pairs if max_pairs is None else pairs[:max_pairs]


preference_pairs = build_preference_pairs()

# DPO requires both responses to extend the exact same tokenized prompt.
for pair in preference_pairs:
    prompt_ids = tokenizer(pair["prompt"], add_special_tokens=False)["input_ids"]
    for response_key in ("chosen", "rejected"):
        combined_ids = tokenizer(
            pair["prompt"] + pair[response_key], add_special_tokens=False
        )["input_ids"]
        if combined_ids[: len(prompt_ids)] != prompt_ids:
            raise ValueError(f"Unstable tokenizer boundary for {response_key} response")

example_preference = preference_pairs[0]
print(
    f"Built {len(preference_pairs)} chat-formatted preference pairs | "
    f"columns: {list(example_preference)}"
)
print("\n=== One offline comparison ===")
print(f"Shared prompt ({len(example_preference['prompt'])} chars):")
print(f"  {example_preference['prompt'][-220:]!r}")
print(f"Chosen response ({len(example_preference['chosen'])} chars):")
print(f"  {example_preference['chosen'][:220]!r}")
print(f"Rejected response ({len(example_preference['rejected'])} chars):")
print(f"  {example_preference['rejected'][:220]!r}")
print("\nThe label is comparative: chosen is preferred to rejected for this same prompt.")

### From Two Toy Responses to Token Sequences

The toy example assigned one probability to each complete response. A language model reaches a response token by token. By the chain rule, its response log-probability is the sum of response-token log-probabilities:

$$
\log\pi(y\mid x)
=
\sum_{t=1}^{T}\log\pi(y_t\mid x,y_{<t}).
$$

The prompt tokens provide context but are excluded from the score. For every preference pair, DPO needs four sequence totals:

| Model | Chosen $y^+$ | Rejected $y^-$ |
| --- | ---: | ---: |
| Trainable policy | $\log\pi_\theta(y^+\mid x)$ | $\log\pi_\theta(y^-\mid x)$ |
| Frozen reference | $\log\pi_{\mathrm{ref}}(y^+\mid x)$ | $\log\pi_{\mathrm{ref}}(y^-\mid x)$ |

Those four numbers become two log-ratio movements, one preference margin, and one scalar loss. This is the sequence-level version of the two-option toy.

Summing token log-probabilities also creates a practical hazard: response length can become a shortcut. Preference data should avoid systematic chosen/rejected length bias, and evaluation should report quality by length slice rather than trusting one aggregate win rate.

The helper below exposes all four scores and the derived DPO terms before `DPOTrainer` performs any optimization.

> **PyTorch → Keras:** TRL's `DPOTrainer` is PyTorch-only. A Keras implementation would run the same policy/reference forward passes, gather response-token log-probabilities, and evaluate the softplus loss inside a custom `train_step`.

In [ ]:
# Step 3: expose the four sequence scores that produce one DPO loss.
def response_sequence_logprob(model, prompt, response):
    """Sum response-token log-probabilities; prompt tokens are context, not targets.

    Summed scores are length-sensitive, so compare length-balanced data and slices.
    """
    prompt_ids = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids
    full_ids = tokenizer(
        prompt + response, add_special_tokens=False, return_tensors="pt"
    ).input_ids

    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + response")

    model_device = next(model.parameters()).device
    full_ids = full_ids.to(model_device)
    model.eval()
    with torch.no_grad():
        logits = model(input_ids=full_ids).logits[:, :-1, :]
        token_logps = F.log_softmax(logits, dim=-1).gather(
            2, full_ids[:, 1:].unsqueeze(-1)
        ).squeeze(-1)

    # A target token at position i is scored by logits from position i - 1.
    response_logps = token_logps[:, prompt_length - 1 :]
    return {
        "sum": response_logps.sum().item(),
        "mean": response_logps.mean().item(),
        "tokens": response_logps.shape[1],
    }


def measure_dpo_pair(policy, reference, pair, beta):
    """Compute four sequence scores, anchored movements, and DPO classification loss."""
    policy_chosen = response_sequence_logprob(policy, pair["prompt"], pair["chosen"])
    policy_rejected = response_sequence_logprob(policy, pair["prompt"], pair["rejected"])
    reference_chosen = response_sequence_logprob(
        reference, pair["prompt"], pair["chosen"]
    )
    reference_rejected = response_sequence_logprob(
        reference, pair["prompt"], pair["rejected"]
    )

    chosen_log_ratio = policy_chosen["sum"] - reference_chosen["sum"]
    rejected_log_ratio = policy_rejected["sum"] - reference_rejected["sum"]
    margin = chosen_log_ratio - rejected_log_ratio
    scaled_margin = beta * margin
    modeled_pair_probability = torch.sigmoid(torch.tensor(scaled_margin)).item()
    loss = F.softplus(torch.tensor(-scaled_margin)).item()

    return {
        "policy_chosen": policy_chosen,
        "policy_rejected": policy_rejected,
        "reference_chosen": reference_chosen,
        "reference_rejected": reference_rejected,
        "chosen_log_ratio": chosen_log_ratio,
        "rejected_log_ratio": rejected_log_ratio,
        "margin": margin,
        "scaled_margin": scaled_margin,
        "modeled_pair_probability": modeled_pair_probability,
        "loss": loss,
    }


def print_dpo_measurement(label, measurement):
    print(f"\n=== {label} ===")
    print("Four response sequence scores (summed log-probability):")
    print(
        f"  policy:    chosen={measurement['policy_chosen']['sum']:+.3f} | "
        f"rejected={measurement['policy_rejected']['sum']:+.3f}"
    )
    print(
        f"  reference: chosen={measurement['reference_chosen']['sum']:+.3f} | "
        f"rejected={measurement['reference_rejected']['sum']:+.3f}"
    )
    print("Response lengths:")
    print(
        f"  chosen={measurement['policy_chosen']['tokens']} tokens | "
        f"rejected={measurement['policy_rejected']['tokens']} tokens"
    )
    print("Movement from frozen SFT (policy minus reference):")
    print(
        f"  chosen={measurement['chosen_log_ratio']:+.3f} | "
        f"rejected={measurement['rejected_log_ratio']:+.3f}"
    )
    print(f"Unscaled movement margin:    {measurement['margin']:+.3f}")
    print(f"Beta-scaled margin:          {measurement['scaled_margin']:+.3f}")
    print(f"DPO-modeled P(chosen wins): {measurement['modeled_pair_probability']:.3f}")
    print(f"DPO loss:                    {measurement['loss']:.3f}")


print("Sequence scorer ready. The next cell measures this same pair before and after DPO training.")

In [ ]:
# Step 4: measure, train, then measure the same DPO terms again.
import gc

from peft import PeftModel
from trl import DPOConfig, DPOTrainer

DPO_BETA = 0.1
DPO_TRACKED_PAIR = preference_pairs[0]

# Policy and reference start from separate, identical SFT snapshots.
dpo_policy_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
dpo_reference_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
dpo_policy_model = PeftModel.from_pretrained(
    dpo_policy_base,
    "./checkpoints/instruction-lora",
    is_trainable=True,
).to(device)
dpo_reference_model = PeftModel.from_pretrained(
    dpo_reference_base,
    "./checkpoints/instruction-lora",
    is_trainable=False,
).to(device)
dpo_reference_model.eval()
for parameter in dpo_reference_model.parameters():
    parameter.requires_grad_(False)

print("Expected before DPO: zero movement, P(chosen wins) = 0.5, loss = log(2).")
before_dpo = measure_dpo_pair(
    dpo_policy_model,
    dpo_reference_model,
    DPO_TRACKED_PAIR,
    DPO_BETA,
)
print_dpo_measurement("Before DPO: identical SFT policy/reference", before_dpo)
assert abs(before_dpo["margin"]) < 1e-4
assert abs(before_dpo["loss"] - torch.log(torch.tensor(2.0)).item()) < 1e-4

dpo_dataset = Dataset.from_list(preference_pairs)
dpo_args = DPOConfig(
    output_dir="./checkpoints/preference-dpo",
    per_device_train_batch_size=1,
    max_steps=DEMO_DPO_STEPS,
    learning_rate=1e-5,
    beta=DPO_BETA,
    logging_steps=10,
    save_strategy="no",
    bf16=False,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_policy_model,
    ref_model=dpo_reference_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)
dpo_trainer.train()

after_dpo = measure_dpo_pair(
    dpo_policy_model,
    dpo_reference_model,
    DPO_TRACKED_PAIR,
    DPO_BETA,
)
print_dpo_measurement("After DPO: same tracked training pair", after_dpo)
print(
    "\nObserved change on this pair:\n"
    f"  margin: {before_dpo['margin']:+.3f} -> {after_dpo['margin']:+.3f}\n"
    f"  loss:   {before_dpo['loss']:.3f} -> {after_dpo['loss']:.3f}"
)
print(
    "This is a mechanism trace on a training pair, not held-out evidence of editor preference."
)

dpo_policy_model.save_pretrained("./checkpoints/preference-dpo")
tokenizer.save_pretrained("./checkpoints/preference-dpo")

# Keep downstream qualitative checks pointed at the aligned policy; release only the frozen copy.
instruct_lora_model = dpo_policy_model
del dpo_trainer, dpo_reference_model, dpo_reference_base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved DPO-aligned adapter from a fresh SFT checkpoint.")

In [ ]:
# Quick health check after DPO


def _dpo_test(label, instruction):
    prompt = render_instruction(instruction)
    print(f"=== {label} ===")
    print(f"  Input : {prompt!r}")
    print(f"  Output: {generate(instruct_lora_model, prompt)}")
    print()


_dpo_test("Test 1: Preferred style", "Who is Aria Voss?")
_dpo_test(
    "Test 2: Still coherent on domain tasks",
    "Continue the fiction narrative: Aria checked the panel and",
)

print("=== Test 3: Mode-collapse check (outputs should vary across attempts) ===")
for attempt in range(3):
    instruction = f"Describe the Meridian (attempt {attempt})."
    prompt = render_instruction(instruction)
    print(f"  [attempt {attempt}] Input : {prompt[:80]!r}...")
    print(f"              Output: {generate(instruct_lora_model, prompt)}")
    print()


### Reading the DPO Trace Honestly

The printed values tell one continuous story:

1. **Four sequence scores:** policy and reference each score chosen and rejected.
2. **Two anchored movements:** subtract reference from policy for each response.
3. **One margin:** chosen movement minus rejected movement.
4. **One modeled pair probability:** $\sigma(\beta\Delta_\theta)$.
5. **One loss:** `softplus(-beta * margin)`.

At initialization, policy and reference are identical, so both movements are zero, the margin is zero, modeled probability is $0.5$, and loss is $\log 2$. Training succeeds mechanically when the chosen response gains more relative ground and the loss falls.

### What beta controls

$\beta$ plays two linked roles. In the RLHF derivation it is the coefficient on KL distance from the SFT reference; in the DPO loss it scales the policy/reference movement margin. Holding a margin fixed, larger $\beta$ makes the modeled pair probability more confident. During fitting, however, it also changes how much policy movement is needed to represent the preference. It is not a universal “quality” dial and must be selected with held-out preference, safety, diversity, and retention checks.

### What this training trace does not prove

The tracked example is a training pair. A better margin on it proves that the implemented objective can move in the labeled direction. It does not prove that editors prefer the model on unseen prompts.

Real preference evaluation still needs:

- held-out prompts and blinded comparisons;
- wins, losses, and ties against the accepted SFT model;
- length-balanced slices to catch shortcut learning;
- instruction, factuality, safety, and diversity gates;
- repeated runs or uncertainty estimates when the decision matters.

### DPO's practical failure modes

- **Coverage ceiling:** offline pairs cannot teach preferences they never contain.
- **Label noise:** inconsistent or weak annotators create contradictory gradients.
- **Length and style shortcuts:** superficial patterns can correlate with `chosen`.
- **Distribution shift:** the trained policy may generate behavior unlike either response in the pair dataset.
- **Over-optimization:** preference improvement can trade away factuality, diversity, or instruction compliance.
- **Reference dependence:** the objective measures movement relative to the selected SFT checkpoint; a weak anchor remains a weak starting point.

### When PPO-based RLHF remains the better tool

Choose PPO or another online RL method when the system must explore new behavior and receives a meaningful reward on generated trajectories—for example, success in an interactive environment, executable tests, game outcomes, or a well-validated process reward. Accept the extra infrastructure only when that online signal offers information fixed offline pairs cannot provide.

For Riverside's current teaching run, DPO is appropriate because the data is an offline set of chosen/rejected responses and the objective itself is what the notebook needs to expose. Production adoption would still depend on the broader evaluation gates developed in Part 3.

---

## Checkpoint Inventory Before Comparison

Part 1 produced three artifacts:

| Artifact | Training signal | Status |
| --- | --- | --- |
| `./checkpoints/non-instruction-full` | Raw manuscript next-token prediction | Full continued-pretraining checkpoint |
| `./checkpoints/instruction-lora` | Prompt/completion demonstrations | SFT LoRA adapter |
| `./checkpoints/preference-dpo` | Chosen/rejected response pairs | Post-SFT DPO adapter; short run remains inconclusive |

The next comparison asks whether their visible behavior matches the objective each practiced. It is not a leaderboard: the objectives, data, and parameter strategies differ.

---

## Same Questions, Four Data-Objective Checkpoints

This is a qualitative recap of the models already trained above. Every candidate receives the same three semantic questions and uses greedy decoding, so sampling noise cannot masquerade as a training effect. Candidates are loaded and released one at a time to keep memory bounded.

Read the columns as **behavior demonstrations, not a leaderboard**: continued pretraining, SFT, and DPO optimize different signals, and the DPO pairs in this teaching run are structural proxies rather than real editor labels. Part 3 supplies the broader evaluation context.


In [ ]:
import gc
import html
import time

from IPython.display import HTML, display
from peft import PeftModel


DATA_COMPARISON_QUESTIONS = {
    "Domain knowledge": "Who is Aria Voss, and what is her role aboard the Meridian's Promise?",
    "Instruction following": (
        "Continue the fiction narrative in the same style.\n\n"
        "Context:\nAria Voss checked the Meridian's Promise status panel and"
    ),
    "Concise editorial style": "In one concise sentence, describe the Meridian's Promise.",
}

DATA_COMPARISON_CANDIDATES = [
    ("Base", "base", MODEL_NAME),
    ("Continued pretraining", "full", "./checkpoints/non-instruction-full"),
    ("SFT LoRA", "adapter", "./checkpoints/instruction-lora"),
    ("DPO after SFT", "adapter", "./checkpoints/preference-dpo"),
]


def load_data_comparison_candidate(kind, model_path):
    """Load one comparison candidate without keeping the other checkpoints in memory."""
    if kind == "base":
        return AutoModelForCausalLM.from_pretrained(model_path).to(device)

    artifact_path = Path(model_path)
    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Missing {artifact_path}. Run the corresponding training cell before this comparison."
        )
    if kind == "full":
        return AutoModelForCausalLM.from_pretrained(artifact_path).to(device)

    adapter_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    return PeftModel.from_pretrained(adapter_base, artifact_path).to(device)


def generate_data_comparison_answer(model, question, max_new_tokens=48):
    """Use deterministic decoding so differences come from checkpoints, not sampling noise."""
    model.eval()
    formatted_prompt = render_instruction(question)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:], skip_special_tokens=True
    ).strip() or "[model stopped immediately]"


data_comparison_answers = {
    question_label: {} for question_label in DATA_COMPARISON_QUESTIONS
}
for candidate_label, candidate_kind, candidate_path in DATA_COMPARISON_CANDIDATES:
    candidate_model = load_data_comparison_candidate(candidate_kind, candidate_path)
    try:
        for question_label, question in DATA_COMPARISON_QUESTIONS.items():
            started = time.perf_counter()
            answer = generate_data_comparison_answer(candidate_model, question)
            elapsed = time.perf_counter() - started
            data_comparison_answers[question_label][candidate_label] = (
                f"{answer}\n\n[{elapsed:.1f}s]"
            )
    finally:
        del candidate_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

header = "".join(
    f"<th style='min-width:190px'>{html.escape(label)}</th>"
    for label, _, _ in DATA_COMPARISON_CANDIDATES
)
body = "".join(
    "<tr>"
    f"<th style='text-align:left;vertical-align:top'>{html.escape(question_label)}</th>"
    + "".join(
        "<td style='vertical-align:top;white-space:pre-wrap'>"
        f"{html.escape(data_comparison_answers[question_label][label])}</td>"
        for label, _, _ in DATA_COMPARISON_CANDIDATES
    )
    + "</tr>"
    for question_label in DATA_COMPARISON_QUESTIONS
)
display(
    HTML(
        "<table><thead><tr><th>Same question</th>"
        + header
        + "</tr></thead><tbody>"
        + body
        + "</tbody></table>"
    )
)


---

## Optional Reference: Production Training Orchestration

The practical objective story is complete: raw manuscripts teach catalog prose, request/response pairs teach an instruction contract, and chosen/rejected pairs teach a comparative preference.

The remaining cells package those stages as separate resumable jobs with checkpointing and explicit model handoffs. Read them when you need orchestration patterns; skip to **Roadmap Checkpoint** if your goal is choosing and evaluating training objectives.

The guarded runner stays disabled because a credible production run needs more than longer training: approved data versions, clean splits, deduplication, repeated runs, independent evaluation, editor review, safety checks, serving measurements, gradual release, and rollback.

The code demonstrates stage boundaries and lineage. It does not turn the short teaching datasets into production-ready models.

In [ ]:
from pathlib import Path
import gc

import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


def _resume_checkpoint(output_path):
    """Return the newest Trainer checkpoint in output_path, if one exists."""
    path = Path(output_path)
    return get_last_checkpoint(str(path)) if path.is_dir() else None


def _release_training_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_continued_pretraining(train_dataset, output_path, max_steps):
    """Continue causal-LM pretraining from a fresh base-model checkpoint."""
    output_path = str(output_path)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=output_path,
            per_device_train_batch_size=2,
            max_steps=max_steps,
            learning_rate=5e-5,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        trainer.save_model(output_path)
        tokenizer.save_pretrained(output_path)
        return dict(result.metrics)
    finally:
        del trainer, model
        _release_training_memory()

### Production SFT: Versioned Adapters

Cloud training jobs usually write LoRA adapters to immutable, versioned artifact storage while keeping the base-model revision pinned separately. Promotion should require evaluation gates, checksum verification, and a rollback pointer to the previous adapter; secrets and raw training text should never be embedded in the adapter directory.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments


def run_lora_sft(train_dataset, output_path, max_steps):
    """Run supervised fine-tuning with a new base model and trainable LoRA adapter."""
    output_path = str(output_path)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model = get_peft_model(
        base_model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            bias="none",
        ),
    )
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=output_path,
            per_device_train_batch_size=2,
            max_steps=max_steps,
            learning_rate=2e-4,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        return dict(result.metrics)
    finally:
        del trainer, model, base_model
        _release_training_memory()

### Production DPO: Controlled Alignment Stage

DPO normally runs as a separate, auditable job from the promoted SFT adapter. Store the preference-dataset version, frozen-reference identity, tokenizer revision, and `beta` with the resulting adapter; gate promotion on held-out preference accuracy, safety checks, and mode-collapse tests.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer


def run_dpo(train_dataset, sft_adapter_path, output_path, max_steps):
    """Align a trainable SFT LoRA policy against an explicit frozen SFT reference."""
    sft_adapter_path = str(sft_adapter_path)
    output_path = str(output_path)
    policy_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    reference_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    policy = PeftModel.from_pretrained(policy_base, sft_adapter_path, is_trainable=True)
    reference = PeftModel.from_pretrained(
        reference_base, sft_adapter_path, is_trainable=False
    )
    reference.eval()
    for parameter in reference.parameters():
        parameter.requires_grad_(False)

    trainer = None
    try:
        args = DPOConfig(
            output_dir=output_path,
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=1e-5,
            beta=0.1,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            bf16=False,
            report_to="none",
        )
        trainer = DPOTrainer(
            model=policy,
            ref_model=reference,
            args=args,
            train_dataset=train_dataset,
            processing_class=tokenizer,
        )
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        policy.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        return dict(result.metrics)
    finally:
        del trainer, policy, reference, policy_base, reference_base
        _release_training_memory()

### Production Orchestration: Build, Train, Gate, Promote

A scheduler or managed training service normally runs these stages with isolated compute, resumable checkpoints, centralized logs, and explicit data/model lineage. The guarded cell below is the local orchestration equivalent: it builds full-corpus datasets and runs the stages in dependency order, but remains off until deliberately enabled.

In [ ]:
import math

RUN_PRODUCTION_PIPELINE = False
PRODUCTION_EPOCHS = 1

production_metrics = {}
if RUN_PRODUCTION_PIPELINE:
    production_root = Path("./checkpoints/production")
    continued_path = production_root / "continued-pretraining"
    sft_path = production_root / "sft-lora"
    dpo_path = production_root / "dpo"

    # Rebuild every training contract from all mapped novels and every chapter.
    all_novel_aliases = list(NOVELS.keys())
    chapter_count = sum(
        len(list((CONTENT_DIR / directory).glob("chapter-*.txt")))
        for directory in NOVELS.values()
    )

    production_paragraphs = load_corpus_paragraphs(
        novels=all_novel_aliases, max_chapters=None
    )
    production_causal = Dataset.from_dict({"text": production_paragraphs}).map(
        lambda examples: tokenize_causal(examples, tokenizer),
        batched=True,
        remove_columns=["text"],
    )

    production_instruction_pairs = build_instruction_pairs(
        novels=all_novel_aliases, max_chapters=None
    )
    production_sft = Dataset.from_list(production_instruction_pairs).map(
        tokenize_instruction, remove_columns=["instruction", "response"]
    )

    production_preference_pairs = build_preference_pairs(
        novels=all_novel_aliases, max_chapters=None, max_pairs=None
    )
    production_dpo = Dataset.from_list(production_preference_pairs)

    print(
        f"Full corpus: {len(all_novel_aliases)} novels, {chapter_count} chapters | "
        f"continued-pretraining chunks={len(production_causal):,}, "
        f"SFT pairs={len(production_sft):,}, DPO pairs={len(production_dpo):,}"
    )

    # Derive max_steps from dataset size so each stage completes full epochs.
    continued_steps = PRODUCTION_EPOCHS * math.ceil(len(production_causal) / 2)
    sft_steps = PRODUCTION_EPOCHS * math.ceil(len(production_sft) / 2)
    dpo_steps = PRODUCTION_EPOCHS * len(production_dpo)

    production_metrics["continued_pretraining"] = run_continued_pretraining(
        train_dataset=production_causal,
        output_path=continued_path,
        max_steps=continued_steps,
    )
    production_metrics["lora_sft"] = run_lora_sft(
        train_dataset=production_sft,
        output_path=sft_path,
        max_steps=sft_steps,
    )
    production_metrics["dpo"] = run_dpo(
        train_dataset=production_dpo,
        sft_adapter_path=sft_path,
        output_path=dpo_path,
        max_steps=dpo_steps,
    )

production_metrics

---

## End of Part 1: The Capability Axis

Each objective addressed a different observed gap:

| Objective | Experience supplied | Capability targeted | Evidence still needed |
| --- | --- | --- | --- |
| Continued pretraining | Raw manuscript next-token prediction | Catalog language and house-style continuation | Clean held-out prose and retention checks |
| SFT | Prompt/completion demonstrations | Bounded instruction following | Representative contract suite |
| DPO | Chosen/rejected response pairs | Relative editor preference | Held-out blinded comparisons against SFT |

The durable intuition is simple:

- raw text changes what prose the model expects;
- demonstrations change what response contract it practices;
- preference pairs change how it ranks already-valid responses.

These artifacts are not a quality leaderboard, and LoRA was only the practical storage choice for two local runs. Continue to **[Part 2: Parameter-Based Techniques](02-llm-finetuning-parameter-techniques.ipynb)** to open that black box and ask how much state must move. Part 3 later reconnects behavior, cost, and workload evidence.